In [30]:
# =========================================================
# Cell 1. 기본 세팅
# =========================================================

# 처음 한 번만 필요할 수 있음
# !pip install pandas numpy statsmodels scikit-learn matplotlib openpyxl pyreadstat

import os
import re
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

print("Setup complete.")

Setup complete.


In [31]:
# =========================================================
# Cell 2. 파일 경로 설정
# =========================================================

# 본인 컴퓨터 경로에 맞게 설정
base_dir = r"C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료"

file_paths = {
    # Compustat annual
    "funda": os.path.join(base_dir, "funda.sas7bdat"),

    # CRSP monthly stock file
    "msf": os.path.join(base_dir, "msf.sas7bdat"),

    # CRSP monthly event/security file
    "mseall": os.path.join(base_dir, "mseall.sas7bdat"),

    # New York Fed CRSP-FRB Link file
    # 교수님이 말한 PERMCO 기준 은행 식별용 파일
    "permco_bank_list": r"C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료\crsp_20240930v2.csv",
}

# 결과 저장 폴더
output_folder = os.path.join(base_dir, "permco_bank_analysis_output")
os.makedirs(output_folder, exist_ok=True)

for name, path in file_paths.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 ** 2)
        print(f"{name}: FOUND | {size_mb:,.2f} MB | {path}")
    else:
        print(f"{name}: NOT FOUND | {path}")

print("\nOutput folder:", output_folder)

funda: FOUND | 6,902.50 MB | C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료\funda.sas7bdat
msf: FOUND | 828.12 MB | C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료\msf.sas7bdat
mseall: FOUND | 1,436.00 MB | C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료\mseall.sas7bdat
permco_bank_list: FOUND | 0.11 MB | C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료\crsp_20240930v2.csv

Output folder: C:\Users\starw\OneDrive\바탕 화면\학교 생활\UNIST\Research\박지훈 자료\permco_bank_analysis_output


In [32]:
# =========================================================
# Cell 3. 공통 함수
# =========================================================

def read_sas_filtered(path, columns=None, chunksize=500_000, filter_func=None, encoding="latin1"):
    """
    대용량 sas7bdat 파일을 chunk 단위로 읽고,
    필요한 컬럼과 조건만 남기는 함수.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    collected = []

    reader = pd.read_sas(
        path,
        format="sas7bdat",
        encoding=encoding,
        chunksize=chunksize
    )

    for i, chunk in enumerate(reader):
        print(f"Reading chunk {i+1}: {chunk.shape}")

        chunk.columns = [str(c) for c in chunk.columns]

        if columns is not None:
            existing_cols = [c for c in columns if c in chunk.columns]
            missing_cols = [c for c in columns if c not in chunk.columns]
            if i == 0 and len(missing_cols) > 0:
                print("Missing columns in this file:", missing_cols)
            chunk = chunk[existing_cols].copy()

        if filter_func is not None:
            chunk = filter_func(chunk)

        if len(chunk) > 0:
            collected.append(chunk)

    if len(collected) == 0:
        print("No rows matched the filter.")
        return pd.DataFrame()

    out = pd.concat(collected, ignore_index=True)
    print("Final loaded shape:", out.shape)
    return out


def safe_divide(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)


def compound_return(ret_series):
    r = pd.to_numeric(ret_series, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan
    return np.prod(1 + r) - 1


def max_drawdown_from_returns(ret_series):
    r = pd.to_numeric(ret_series, errors="coerce").dropna()
    if len(r) == 0:
        return np.nan

    wealth = (1 + r).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1
    return drawdown.min()


def winsorize_series(s, lower=0.01, upper=0.99):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() < 20:
        return s
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lo, hi)


def make_stars(p):
    if pd.isna(p):
        return ""
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""


def clean_permco(x):
    """
    PERMCO를 숫자형 정수로 정리.
    """
    return pd.to_numeric(x, errors="coerce").astype("Int64")


print("Common functions ready.")

Common functions ready.


In [33]:
# =========================================================
# Cell 4. New York Fed CRSP-FRB Link에서 은행 PERMCO 리스트 만들기
# =========================================================

def parse_yyyymmdd(x):
    """
    20240930 같은 YYYYMMDD 숫자/문자열을 datetime으로 변환.
    변환 불가능하면 NaT.
    """
    return pd.to_datetime(x.astype(str), format="%Y%m%d", errors="coerce")


def load_crsp_frb_permco_link(
    path,
    analysis_start="2021-01-01",
    analysis_end="2023-12-31",
    keep_only_bank_related_inst_types=False
):
    """
    New York Fed CRSP-FRB Link 파일에서 PERMCO 은행 리스트 생성.

    네 파일 구조:
    - name
    - inst_type
    - entity
    - permco
    - dt_start
    - dt_end

    기본 원칙:
    1. permco를 CRSP 회사 단위 은행 식별자로 사용
    2. dt_start, dt_end를 사용해 2021-2023 연구기간과 겹치는 link만 유지
    3. PERMCO 중복 제거
    """

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    df = pd.read_csv(path)

    # 컬럼명 정리
    df.columns = [str(c).strip().lower() for c in df.columns]

    print("Raw CRSP-FRB Link shape:", df.shape)
    print("Columns:", df.columns.tolist())

    required_cols = ["name", "inst_type", "entity", "permco", "dt_start", "dt_end"]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # 기본 정리
    df["permco"] = pd.to_numeric(df["permco"], errors="coerce")
    df["entity"] = pd.to_numeric(df["entity"], errors="coerce")

    df = df.dropna(subset=["permco"]).copy()
    df["permco"] = df["permco"].astype(int)
    df["PERMCO"] = df["permco"]

    # 날짜 변환
    df["link_start"] = parse_yyyymmdd(df["dt_start"])
    df["link_end"] = parse_yyyymmdd(df["dt_end"])

    analysis_start = pd.Timestamp(analysis_start)
    analysis_end = pd.Timestamp(analysis_end)

    print("\nOriginal link date range:")
    print("link_start min:", df["link_start"].min())
    print("link_start max:", df["link_start"].max())
    print("link_end min:", df["link_end"].min())
    print("link_end max:", df["link_end"].max())

    print("\nInstitution type distribution, raw:")
    display(df["inst_type"].value_counts(dropna=False).reset_index().rename(
        columns={"index": "inst_type", "inst_type": "N"}
    ))

    # 선택 옵션: 명백한 은행 관련 inst_type만 유지
    # 기본값은 False. 교수님이 "링크에 있는 PERMCO"라고 했으므로 일단 전체 link 사용 권장.
    if keep_only_bank_related_inst_types:
        bank_keywords = [
            "bank",
            "thrift",
            "fsb",
            "national",
            "member"
        ]

        pattern = "|".join(bank_keywords)

        before = len(df)
        df = df[
            df["inst_type"]
            .astype(str)
            .str.lower()
            .str.contains(pattern, na=False)
        ].copy()

        print(f"\nFiltered to bank-related inst_type: {before:,} -> {len(df):,}")

    # 2021-2023 연구기간과 link 기간이 겹치는 PERMCO만 유지
    before = len(df)

    df_active = df[
        (df["link_start"].isna() | (df["link_start"] <= analysis_end)) &
        (df["link_end"].isna() | (df["link_end"] >= analysis_start))
    ].copy()

    print(f"\nApplied active-link filter for {analysis_start.date()} to {analysis_end.date()}:")
    print(f"{before:,} rows -> {len(df_active):,} rows")

    # PERMCO 중복 제거
    # 같은 PERMCO가 여러 entity/RSSD에 연결될 수 있으므로,
    # 연구기간과 겹치는 link 중 가장 최근까지 유효한 link를 대표로 선택.
    bank_permco_list = (
        df_active
        .sort_values(["permco", "link_end", "link_start"], ascending=[True, False, False])
        .drop_duplicates(subset=["permco"], keep="first")
        .copy()
    )

    print("\nUnique PERMCO count:")
    print("Raw file:", df["permco"].nunique())
    print("Active during analysis window:", df_active["permco"].nunique())
    print("After one-row-per-PERMCO:", bank_permco_list["PERMCO"].nunique())

    print("\nInstitution type distribution, active links:")
    display(df_active["inst_type"].value_counts(dropna=False).reset_index().rename(
        columns={"index": "inst_type", "inst_type": "N"}
    ))

    print("\nSample of final PERMCO bank list:")
    display(bank_permco_list.head(20))

    return bank_permco_list, df_active, df


# 실행
bank_permco_list_raw, bank_permco_link_filtered, bank_permco_link_all = load_crsp_frb_permco_link(
    file_paths["permco_bank_list"],
    analysis_start="2021-01-01",
    analysis_end="2023-12-31",
    keep_only_bank_related_inst_types=False
)

# 이후 셀에서 사용할 PERMCO 리스트
bank_permcos = sorted(
    bank_permco_list_raw["PERMCO"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

bank_permco_set = set(bank_permcos)

print("\nNumber of bank PERMCOs used in analysis:", len(bank_permcos))
print("First 20 PERMCOs:", bank_permcos[:20])

Raw CRSP-FRB Link shape: (1495, 6)
Columns: ['name', 'inst_type', 'entity', 'permco', 'dt_start', 'dt_end']

Original link date range:
link_start min: 1986-06-30 00:00:00
link_start max: 2024-07-12 00:00:00
link_end min: 1986-10-31 00:00:00
link_end max: 2024-09-30 00:00:00

Institution type distribution, raw:


,N,count
0,Bank Holding Company,1213
1,Commercial Bank,127
2,Thrift holding company,83
3,Thrift Holding Company,40
4,Domestic Entity Other,11
5,Bank holding company,10
6,Non Member Bank,6
7,Thrift,1
8,NaN,1
9,Thirft Holding Company,1



Applied active-link filter for 2021-01-01 to 2023-12-31:
1,495 rows -> 395 rows

Unique PERMCO count:
Raw file: 1450
Active during analysis window: 394
After one-row-per-PERMCO: 394

Institution type distribution, active links:


,N,count
0,Bank Holding Company,320
1,Thrift Holding Company,32
2,Thrift holding company,24
3,Commercial Bank,6
4,Non Member Bank,5
5,Domestic Entity Other,2
6,Bank holding company,2
7,NaN,1
8,Thirft Holding Company,1
9,FSB,1



Sample of final PERMCO bank list:


,name,inst_type,entity,permco,dt_start,dt_end,PERMCO,link_start,link_end
1214,AMERICAN EXPRESS CO,Thrift Holding Company,1275216,90,19860630,20240930,90,1986-06-30,2024-09-30
1108,AMERICAN INTERNATIONAL GROUP,Thrift holding company,1562176,137,19860630,20240930,137,1986-06-30,2024-09-30
1215,ASSOCIATED BANC-CORP,Bank Holding Company,1199563,362,19860630,20240930,362,1986-06-30,2024-09-30
1109,"POPULAR, INC.",Bank Holding Company,1129382,475,19860630,20240930,475,1986-06-30,2024-09-30
1487,NEWTEKONE INC,Bank Holding Company,5586741,579,20230106,20240930,579,2023-01-06,2024-09-30
1216,BANK OF HAWAII CORP,Bank Holding Company,1025309,589,19860630,20240930,589,1986-06-30,2024-09-30
1217,COMMERCE BANCSHARES INC,Bank Holding Company,1049341,779,19860630,20240930,779,1986-06-30,2024-09-30
1218,SYNOVUS FINANCIAL CORP,Bank Holding Company,1078846,781,19860630,20240930,781,1986-06-30,2024-09-30
1219,CULLEN/FROST BANKERS INC,Bank Holding Company,1102367,840,19860630,20240930,840,1986-06-30,2024-09-30
1220,COMERICA INC,Bank Holding Company,1199844,1261,19860630,20240930,1261,1986-06-30,2024-09-30



Number of bank PERMCOs used in analysis: 394
First 20 PERMCOs: [90, 137, 362, 475, 579, 589, 779, 781, 840, 1261, 1620, 1645, 1658, 1689, 1741, 1856, 1940, 2093, 2253, 2535]


In [34]:
# =========================================================
# Cell 5. 분석 기간 설정
# =========================================================

# 메인 분석 기간
main_start = pd.Timestamp("2022-03-01")
main_end = pd.Timestamp("2023-03-31")

# 확장 분석 기간
extended_start = pd.Timestamp("2022-03-01")
extended_end = pd.Timestamp("2023-07-31")

# March 2023 stress window
march_start = pd.Timestamp("2023-03-01")
march_end = pd.Timestamp("2023-03-31")

# CRSP에서 읽을 전체 기간
shock_start = main_start
shock_end = extended_end

print("Main period:", main_start, "to", main_end)
print("Extended period:", extended_start, "to", extended_end)
print("March 2023:", march_start, "to", march_end)

Main period: 2022-03-01 00:00:00 to 2023-03-31 00:00:00
Extended period: 2022-03-01 00:00:00 to 2023-07-31 00:00:00
March 2023: 2023-03-01 00:00:00 to 2023-03-31 00:00:00


In [35]:
# =========================================================
# Cell 6. CRSP msf에서 PERMCO 은행 수익률 추출
# =========================================================

msf_cols = [
    "CUSIP", "PERMNO", "PERMCO", "DATE",
    "RET", "RETX", "PRC", "SHROUT", "VOL",
    "HSICCD", "HEXCD"
]

bank_permco_set = set(bank_permcos)

def filter_msf_permco_banks(chunk):
    chunk.columns = [str(c) for c in chunk.columns]

    chunk["DATE"] = pd.to_datetime(chunk["DATE"], errors="coerce")
    chunk["PERMCO"] = pd.to_numeric(chunk["PERMCO"], errors="coerce")

    chunk = chunk[
        (chunk["DATE"] >= shock_start) &
        (chunk["DATE"] <= shock_end) &
        (chunk["PERMCO"].isin(bank_permco_set))
    ].copy()

    return chunk


msf_bank_raw = read_sas_filtered(
    file_paths["msf"],
    columns=msf_cols,
    chunksize=500_000,
    filter_func=filter_msf_permco_banks
)

msf_bank = msf_bank_raw.copy()

if len(msf_bank) == 0:
    raise ValueError("No CRSP MSF rows matched the PERMCO bank list. PERMCO list or CRSP file should be checked.")

# 변수 정리
msf_bank["CUSIP"] = msf_bank["CUSIP"].astype(str).str[:8]
msf_bank["PERMNO"] = pd.to_numeric(msf_bank["PERMNO"], errors="coerce")
msf_bank["PERMCO"] = pd.to_numeric(msf_bank["PERMCO"], errors="coerce")
msf_bank["RET"] = pd.to_numeric(msf_bank["RET"], errors="coerce")
msf_bank["RETX"] = pd.to_numeric(msf_bank["RETX"], errors="coerce")
msf_bank["PRC"] = pd.to_numeric(msf_bank["PRC"], errors="coerce")
msf_bank["SHROUT"] = pd.to_numeric(msf_bank["SHROUT"], errors="coerce")
msf_bank["VOL"] = pd.to_numeric(msf_bank["VOL"], errors="coerce")

# CRSP PRC는 bid/ask average 표시 때문에 음수일 수 있으므로 시가총액 계산에는 절댓값 사용
msf_bank["abs_prc"] = msf_bank["PRC"].abs()
msf_bank["mktcap_crsp"] = msf_bank["abs_prc"] * msf_bank["SHROUT"]

print("msf_bank shape:", msf_bank.shape)
print("Unique PERMCO:", msf_bank["PERMCO"].nunique())
print("Unique PERMNO:", msf_bank["PERMNO"].nunique())
print("Date range:", msf_bank["DATE"].min(), "to", msf_bank["DATE"].max())

display(msf_bank.head())

Reading chunk 1: (500000, 21)
Reading chunk 2: (500000, 21)
Reading chunk 3: (500000, 21)
Reading chunk 4: (500000, 21)
Reading chunk 5: (500000, 21)
Reading chunk 6: (500000, 21)
Reading chunk 7: (500000, 21)
Reading chunk 8: (500000, 21)
Reading chunk 9: (500000, 21)
Reading chunk 10: (500000, 21)
Reading chunk 11: (153763, 21)
Final loaded shape: (5923, 11)
msf_bank shape: (5923, 13)
Unique PERMCO: 362
Unique PERMNO: 362
Date range: 2022-03-31 00:00:00 to 2023-07-31 00:00:00


,CUSIP,PERMNO,PERMCO,DATE,RET,RETX,PRC,SHROUT,VOL,HSICCD,HEXCD,abs_prc,mktcap_crsp
0,74144T10,"10,138.00000","8,087.00000",2022-03-31,0.05416,0.04586,151.19000,"227,810.00000","380,316.00000","6,211.00000",3.00000,151.19000,"34,442,594.45618"
1,74144T10,"10,138.00000","8,087.00000",2022-04-29,-0.18619,-0.18619,123.04000,"227,297.00000","325,938.00000","6,211.00000",3.00000,123.04000,"27,966,623.08810"
2,74144T10,"10,138.00000","8,087.00000",2022-05-31,0.03292,0.03292,127.09000,"227,297.00000","386,381.00000","6,211.00000",3.00000,127.09000,"28,887,174.89761"
3,74144T10,"10,138.00000","8,087.00000",2022-06-30,-0.09662,-0.10607,113.61000,"225,715.00000","376,969.00000","6,211.00000",3.00000,113.61000,"25,643,481.28777"
4,74144T10,"10,138.00000","8,087.00000",2022-07-29,0.08679,0.08679,123.47000,"225,692.00000","308,361.00000","6,211.00000",3.00000,123.47000,"27,866,191.51550"


In [36]:
# =========================================================
# Cell 7. CRSP mseall에서 보조정보 및 delisting return 추출
# =========================================================

mseall_cols = [
    "DATE", "CUSIP", "NCUSIP", "COMNAM", "TICKER",
    "PERMNO", "PERMCO",
    "SHRCD", "EXCHCD", "SICCD", "NAICS",
    "TRDSTAT", "SECSTAT",
    "DLRET", "DLRETX", "DLSTCD"
]

def filter_mseall_permco_banks(chunk):
    chunk.columns = [str(c) for c in chunk.columns]

    chunk["DATE"] = pd.to_datetime(chunk["DATE"], errors="coerce")
    chunk["PERMCO"] = pd.to_numeric(chunk["PERMCO"], errors="coerce")

    chunk = chunk[
        (chunk["DATE"] >= shock_start) &
        (chunk["DATE"] <= shock_end) &
        (chunk["PERMCO"].isin(bank_permco_set))
    ].copy()

    return chunk


mseall_bank_raw = read_sas_filtered(
    file_paths["mseall"],
    columns=mseall_cols,
    chunksize=500_000,
    filter_func=filter_mseall_permco_banks
)

mseall_bank = mseall_bank_raw.copy()

if len(mseall_bank) > 0:
    mseall_bank["CUSIP"] = mseall_bank["CUSIP"].astype(str).str[:8]
    mseall_bank["NCUSIP"] = mseall_bank["NCUSIP"].astype(str).str[:8]

    for col in ["PERMNO", "PERMCO", "SHRCD", "EXCHCD", "SICCD", "NAICS", "DLRET", "DLRETX", "DLSTCD"]:
        if col in mseall_bank.columns:
            mseall_bank[col] = pd.to_numeric(mseall_bank[col], errors="coerce")

    # 월별 delisting return
    dlret_monthly = (
        mseall_bank[
            ["PERMNO", "PERMCO", "DATE", "DLRET", "DLRETX", "DLSTCD"]
        ]
        .drop_duplicates(subset=["PERMNO", "PERMCO", "DATE"])
        .copy()
    )

    # PERMNO-DATE별 보조정보
    crsp_monthly_info = (
        mseall_bank
        .sort_values(["PERMNO", "DATE"])
        .drop_duplicates(subset=["PERMNO", "PERMCO", "DATE"], keep="last")
        [[
            "PERMNO", "PERMCO", "DATE", "CUSIP", "NCUSIP",
            "COMNAM", "TICKER", "SHRCD", "EXCHCD", "SICCD", "NAICS",
            "TRDSTAT", "SECSTAT"
        ]]
        .copy()
    )

    # PERMCO별 가장 최근 회사정보
    crsp_permco_info = (
        mseall_bank
        .sort_values(["PERMCO", "DATE"])
        .drop_duplicates(subset=["PERMCO"], keep="last")
        [[
            "PERMCO", "COMNAM", "TICKER", "SICCD", "NAICS",
            "TRDSTAT", "SECSTAT"
        ]]
        .copy()
    )

else:
    dlret_monthly = pd.DataFrame()
    crsp_monthly_info = pd.DataFrame()
    crsp_permco_info = pd.DataFrame()

print("mseall_bank shape:", mseall_bank.shape)
print("dlret_monthly shape:", dlret_monthly.shape)
print("crsp_monthly_info shape:", crsp_monthly_info.shape)
print("crsp_permco_info shape:", crsp_permco_info.shape)

display(crsp_permco_info.head())

Reading chunk 1: (500000, 50)
Reading chunk 2: (500000, 50)
Reading chunk 3: (500000, 50)
Reading chunk 4: (500000, 50)
Reading chunk 5: (500000, 50)
Reading chunk 6: (500000, 50)
Reading chunk 7: (500000, 50)
Reading chunk 8: (244882, 50)
Final loaded shape: (5739, 16)
mseall_bank shape: (5739, 16)
dlret_monthly shape: (5714, 6)
crsp_monthly_info shape: (5714, 13)
crsp_permco_info shape: (363, 7)


,PERMCO,COMNAM,TICKER,SICCD,NAICS,TRDSTAT,SECSTAT
2495,90.00000,AMERICAN EXPRESS CO,AXP,"6,141.00000","522,291.00000",A,R
2580,137.00000,AMERICAN INTERNATIONAL GROUP INC,AIG,"6,331.00000","524,126.00000",A,R
943,362.00000,ASSOCIATED BANC CORP,ASB,"6,022.00000","522,110.00000",A,R
1144,475.00000,POPULAR INC,BPOP,"6,022.00000","522,110.00000",A,R
2478,579.00000,NEWTEKONE INC,NEWT,"2,330.00000","522,291.00000",A,R


In [37]:
# =========================================================
# Cell 8. Delisting-adjusted return 생성
# =========================================================

msf_adj = msf_bank.copy()

if len(dlret_monthly) > 0:
    msf_adj = msf_adj.merge(
        dlret_monthly,
        on=["PERMNO", "PERMCO", "DATE"],
        how="left"
    )
else:
    msf_adj["DLRET"] = np.nan
    msf_adj["DLRETX"] = np.nan
    msf_adj["DLSTCD"] = np.nan

# RET + DLRET 결합
msf_adj["ret_adj"] = np.nan

cond_ret_only = msf_adj["RET"].notna() & msf_adj["DLRET"].isna()
cond_both = msf_adj["RET"].notna() & msf_adj["DLRET"].notna()
cond_dlret_only = msf_adj["RET"].isna() & msf_adj["DLRET"].notna()

msf_adj.loc[cond_ret_only, "ret_adj"] = msf_adj.loc[cond_ret_only, "RET"]
msf_adj.loc[cond_both, "ret_adj"] = (
    (1 + msf_adj.loc[cond_both, "RET"]) *
    (1 + msf_adj.loc[cond_both, "DLRET"]) - 1
)
msf_adj.loc[cond_dlret_only, "ret_adj"] = msf_adj.loc[cond_dlret_only, "DLRET"]

# RETX + DLRETX도 보조로 생성
msf_adj["retx_adj"] = np.nan

cond_retx_only = msf_adj["RETX"].notna() & msf_adj["DLRETX"].isna()
cond_retx_both = msf_adj["RETX"].notna() & msf_adj["DLRETX"].notna()
cond_dlretx_only = msf_adj["RETX"].isna() & msf_adj["DLRETX"].notna()

msf_adj.loc[cond_retx_only, "retx_adj"] = msf_adj.loc[cond_retx_only, "RETX"]
msf_adj.loc[cond_retx_both, "retx_adj"] = (
    (1 + msf_adj.loc[cond_retx_both, "RETX"]) *
    (1 + msf_adj.loc[cond_retx_both, "DLRETX"]) - 1
)
msf_adj.loc[cond_dlretx_only, "retx_adj"] = msf_adj.loc[cond_dlretx_only, "DLRETX"]

print("msf_adj shape:", msf_adj.shape)
print("ret_adj missing ratio:", msf_adj["ret_adj"].isna().mean())
print("Rows with DLRET:", msf_adj["DLRET"].notna().sum())

display(msf_adj[["PERMCO", "PERMNO", "DATE", "RET", "DLRET", "ret_adj", "PRC", "SHROUT", "mktcap_crsp"]].head())

msf_adj shape: (5923, 18)
ret_adj missing ratio: 0.0016883336147222692
Rows with DLRET: 24


,PERMCO,PERMNO,DATE,RET,DLRET,ret_adj,PRC,SHROUT,mktcap_crsp
0,"8,087.00000","10,138.00000",2022-03-31,0.05416,NaN,0.05416,151.19000,"227,810.00000","34,442,594.45618"
1,"8,087.00000","10,138.00000",2022-04-29,-0.18619,NaN,-0.18619,123.04000,"227,297.00000","27,966,623.08810"
2,"8,087.00000","10,138.00000",2022-05-31,0.03292,NaN,0.03292,127.09000,"227,297.00000","28,887,174.89761"
3,"8,087.00000","10,138.00000",2022-06-30,-0.09662,NaN,-0.09662,113.61000,"225,715.00000","25,643,481.28777"
4,"8,087.00000","10,138.00000",2022-07-29,0.08679,NaN,0.08679,123.47000,"225,692.00000","27,866,191.51550"


In [38]:
# =========================================================
# Cell 9. 보통주 및 주요 거래소 필터 적용
# =========================================================

crsp_returns = msf_adj.copy()

# 월별 CRSP 보조정보 붙이기
if len(crsp_monthly_info) > 0:
    info_to_merge = crsp_monthly_info.drop(columns=["CUSIP"], errors="ignore")
    crsp_returns = crsp_returns.merge(
        info_to_merge,
        on=["PERMNO", "PERMCO", "DATE"],
        how="left"
    )

print("Before filters:", crsp_returns.shape)

# 보통주만 유지: SHRCD 10, 11
if "SHRCD" in crsp_returns.columns:
    before = len(crsp_returns)
    crsp_returns = crsp_returns[
        crsp_returns["SHRCD"].isna() |
        crsp_returns["SHRCD"].isin([10, 11])
    ].copy()
    print(f"SHRCD common share filter: {before:,} -> {len(crsp_returns):,}")

# 주요 거래소만 유지: NYSE, AMEX, NASDAQ
if "EXCHCD" in crsp_returns.columns:
    before = len(crsp_returns)
    crsp_returns = crsp_returns[
        crsp_returns["EXCHCD"].isna() |
        crsp_returns["EXCHCD"].isin([1, 2, 3])
    ].copy()
    print(f"EXCHCD filter: {before:,} -> {len(crsp_returns):,}")

# 수익률 없는 행 제거
before = len(crsp_returns)
crsp_returns = crsp_returns.dropna(subset=["ret_adj"]).copy()
print(f"Drop missing ret_adj: {before:,} -> {len(crsp_returns):,}")

print("Final CRSP return data:", crsp_returns.shape)
print("Unique PERMCO:", crsp_returns["PERMCO"].nunique())
print("Unique PERMNO:", crsp_returns["PERMNO"].nunique())

display(crsp_returns.head())

Before filters: (5923, 27)
SHRCD common share filter: 5,923 -> 5,908
EXCHCD filter: 5,908 -> 5,908
Drop missing ret_adj: 5,908 -> 5,898
Final CRSP return data: (5898, 27)
Unique PERMCO: 362
Unique PERMNO: 362


,CUSIP,PERMNO,PERMCO,DATE,RET,RETX,PRC,SHROUT,VOL,HSICCD,HEXCD,abs_prc,mktcap_crsp,DLRET,DLRETX,DLSTCD,ret_adj,retx_adj,NCUSIP,COMNAM,TICKER,SHRCD,EXCHCD,SICCD,NAICS,TRDSTAT,SECSTAT
0,74144T10,"10,138.00000","8,087.00000",2022-03-31,0.05416,0.04586,151.19000,"227,810.00000","380,316.00000","6,211.00000",3.00000,151.19000,"34,442,594.45618",NaN,NaN,NaN,0.05416,0.04586,74144T10,T ROWE PRICE GROUP INC,TROW,11.00000,3.00000,"6,211.00000","523,930.00000",A,R
1,74144T10,"10,138.00000","8,087.00000",2022-04-29,-0.18619,-0.18619,123.04000,"227,297.00000","325,938.00000","6,211.00000",3.00000,123.04000,"27,966,623.08810",NaN,NaN,NaN,-0.18619,-0.18619,74144T10,T ROWE PRICE GROUP INC,TROW,11.00000,3.00000,"6,211.00000","523,930.00000",A,R
2,74144T10,"10,138.00000","8,087.00000",2022-05-31,0.03292,0.03292,127.09000,"227,297.00000","386,381.00000","6,211.00000",3.00000,127.09000,"28,887,174.89761",NaN,NaN,NaN,0.03292,0.03292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,74144T10,"10,138.00000","8,087.00000",2022-06-30,-0.09662,-0.10607,113.61000,"225,715.00000","376,969.00000","6,211.00000",3.00000,113.61000,"25,643,481.28777",NaN,NaN,NaN,-0.09662,-0.10607,74144T10,T ROWE PRICE GROUP INC,TROW,11.00000,3.00000,"6,211.00000","523,930.00000",A,R
4,74144T10,"10,138.00000","8,087.00000",2022-07-29,0.08679,0.08679,123.47000,"225,692.00000","308,361.00000","6,211.00000",3.00000,123.47000,"27,866,191.51550",NaN,NaN,NaN,0.08679,0.08679,74144T10,T ROWE PRICE GROUP INC,TROW,11.00000,3.00000,"6,211.00000","523,930.00000",A,R


In [39]:
# =========================================================
# Cell 10. PERMCO-month 단위 수익률 생성
# =========================================================

# 한 PERMCO-DATE에 여러 PERMNO가 있으면 mktcap_crsp가 가장 큰 security 선택
permco_monthly = (
    crsp_returns
    .sort_values(
        ["PERMCO", "DATE", "mktcap_crsp"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["PERMCO", "DATE"], keep="first")
    .copy()
)

print("permco_monthly shape:", permco_monthly.shape)
print("Unique PERMCO:", permco_monthly["PERMCO"].nunique())
print("Unique PERMNO:", permco_monthly["PERMNO"].nunique())

# PERMCO별 몇 개 PERMNO가 있었는지 진단
permno_count_by_permco = (
    crsp_returns
    .groupby("PERMCO")["PERMNO"]
    .nunique()
    .reset_index(name="n_permno")
    .sort_values("n_permno", ascending=False)
)

print("PERMCOs with multiple PERMNOs:")
display(permno_count_by_permco.head(20))

display(permco_monthly[["PERMCO", "PERMNO", "DATE", "CUSIP", "ret_adj", "mktcap_crsp"]].head())

permco_monthly shape: (5898, 27)
Unique PERMCO: 362
Unique PERMNO: 362
PERMCOs with multiple PERMNOs:


,PERMCO,n_permno
0,90.00000,1
1,137.00000,1
2,362.00000,1
3,475.00000,1
4,579.00000,1
5,589.00000,1
6,779.00000,1
7,781.00000,1
8,840.00000,1
9,"1,261.00000",1


,PERMCO,PERMNO,DATE,CUSIP,ret_adj,mktcap_crsp
2553,90.00000,"59,176.00000",2022-03-31,02581610,-0.03876,"141,613,043.00000"
2554,90.00000,"59,176.00000",2022-04-29,02581610,-0.06294,"131,567,117.65594"
2555,90.00000,"59,176.00000",2022-05-31,02581610,-0.03371,"127,131,594.71558"
2556,90.00000,"59,176.00000",2022-06-30,02581610,-0.17581,"104,389,173.52295"
2557,90.00000,"59,176.00000",2022-07-29,02581610,0.11110,"115,476,190.16327"


In [40]:
# =========================================================
# Cell 11. PERMCO 기준 은행 벤치마크 수익률 생성
# =========================================================

monthly_for_bench = permco_monthly.copy()

# Equal-weight bank benchmark
ew_bench = (
    monthly_for_bench
    .groupby("DATE", as_index=False)
    .agg(ew_bank_ret=("ret_adj", "mean"))
)

# Value-weight bank benchmark
def value_weighted_return(group):
    temp = group.dropna(subset=["ret_adj", "mktcap_crsp"]).copy()
    temp = temp[temp["mktcap_crsp"] > 0]
    if len(temp) == 0:
        return np.nan
    return np.average(temp["ret_adj"], weights=temp["mktcap_crsp"])

vw_bench = (
    monthly_for_bench
    .groupby("DATE")
    .apply(value_weighted_return)
    .reset_index(name="vw_bank_ret")
)

bank_bench = ew_bench.merge(vw_bench, on="DATE", how="outer")

permco_monthly = permco_monthly.merge(
    bank_bench,
    on="DATE",
    how="left"
)

# 월별 abnormal return
permco_monthly["ew_bank_adj_ret"] = permco_monthly["ret_adj"] - permco_monthly["ew_bank_ret"]
permco_monthly["vw_bank_adj_ret"] = permco_monthly["ret_adj"] - permco_monthly["vw_bank_ret"]

print("bank_bench:")
display(bank_bench)

display(
    permco_monthly[
        ["PERMCO", "PERMNO", "DATE", "ret_adj", "ew_bank_ret", "vw_bank_ret", "ew_bank_adj_ret", "vw_bank_adj_ret"]
    ].head()
)

bank_bench:


,DATE,ew_bank_ret,vw_bank_ret
0,2022-03-31,-0.02689,-0.05137
1,2022-04-29,-0.06518,-0.10336
2,2022-05-31,0.02523,0.05753
3,2022-06-30,-0.06568,-0.12246
4,2022-07-29,0.06044,0.08457
5,2022-08-31,-0.01440,-0.01071
6,2022-09-30,-0.04872,-0.07672
7,2022-10-31,0.09019,0.13341
8,2022-11-30,0.02458,0.06155
9,2022-12-30,-0.04811,-0.06809


,PERMCO,PERMNO,DATE,ret_adj,ew_bank_ret,vw_bank_ret,ew_bank_adj_ret,vw_bank_adj_ret
0,90.00000,"59,176.00000",2022-03-31,-0.03876,-0.02689,-0.05137,-0.01187,0.01262
1,90.00000,"59,176.00000",2022-04-29,-0.06294,-0.06518,-0.10336,0.00224,0.04042
2,90.00000,"59,176.00000",2022-05-31,-0.03371,0.02523,0.05753,-0.05894,-0.09125
3,90.00000,"59,176.00000",2022-06-30,-0.17581,-0.06568,-0.12246,-0.11012,-0.05335
4,90.00000,"59,176.00000",2022-07-29,0.11110,0.06044,0.08457,0.05066,0.02652


In [41]:
# =========================================================
# Cell 12. 기간별 종속변수 생성
# =========================================================

periods = {
    "202203_202303": (pd.Timestamp("2022-03-01"), pd.Timestamp("2023-03-31"), 10),
    "202203_202307": (pd.Timestamp("2022-03-01"), pd.Timestamp("2023-07-31"), 13),
    "202303": (pd.Timestamp("2023-03-01"), pd.Timestamp("2023-03-31"), 1)
}

def make_y_by_period_permco(df, period_name, start, end, min_months):
    temp = df[
        (df["DATE"] >= start) &
        (df["DATE"] <= end)
    ].copy()

    y = (
        temp
        .groupby("PERMCO", as_index=False)
        .agg(
            raw_cum_ret=("ret_adj", compound_return),
            ew_adj_cum_ret=("ew_bank_adj_ret", compound_return),
            vw_adj_cum_ret=("vw_bank_adj_ret", compound_return),
            max_drawdown=("ret_adj", max_drawdown_from_returns),
            n_months=("ret_adj", lambda x: pd.to_numeric(x, errors="coerce").notna().sum()),
            first_date=("DATE", "min"),
            last_date=("DATE", "max"),
            avg_mktcap_crsp=("mktcap_crsp", "mean"),
            representative_permno=("PERMNO", "first"),
            representative_cusip=("CUSIP", "first")
        )
    )

    y = y[y["n_months"] >= min_months].copy()

    rename_dict = {
        "raw_cum_ret": f"raw_cum_ret_{period_name}",
        "ew_adj_cum_ret": f"ew_adj_cum_ret_{period_name}",
        "vw_adj_cum_ret": f"vw_adj_cum_ret_{period_name}",
        "max_drawdown": f"max_drawdown_{period_name}",
        "n_months": f"n_months_{period_name}",
        "first_date": f"first_date_{period_name}",
        "last_date": f"last_date_{period_name}",
        "avg_mktcap_crsp": f"avg_mktcap_crsp_{period_name}",
        "representative_permno": f"representative_permno_{period_name}",
        "representative_cusip": f"representative_cusip_{period_name}"
    }

    y = y.rename(columns=rename_dict)
    return y


y_data = {}

for period_name, (start, end, min_months) in periods.items():
    y_data[period_name] = make_y_by_period_permco(
        permco_monthly,
        period_name,
        start,
        end,
        min_months
    )
    print(period_name, y_data[period_name].shape)
    display(y_data[period_name].head())

y_main = y_data["202203_202303"]
y_extended = y_data["202203_202307"]
y_march = y_data["202303"]

202203_202303 (345, 11)


,PERMCO,raw_cum_ret_202203_202303,ew_adj_cum_ret_202203_202303,vw_adj_cum_ret_202203_202303,max_drawdown_202203_202303,n_months_202203_202303,first_date_202203_202303,last_date_202203_202303,avg_mktcap_crsp_202203_202303,representative_permno_202203_202303,representative_cusip_202203_202303
0,90.00000,-0.14085,0.09676,0.05586,-0.27369,13,2022-03-31,2023-03-31,"119,794,186.07620","59,176.00000",02581610
1,137.00000,-0.15311,0.09152,0.03360,-0.23372,13,2022-03-31,2023-03-31,"43,703,519.26920","66,800.00000",02687478
2,362.00000,-0.23497,0.00777,-0.04623,-0.26248,13,2022-03-31,2023-03-31,"3,205,858.47069","15,318.00000",04548710
3,475.00000,-0.34995,-0.17326,-0.22550,-0.27443,13,2022-03-31,2023-03-31,"5,442,296.99138","16,505.00000",73317470
5,589.00000,-0.37375,-0.17540,-0.22974,-0.35689,13,2022-03-31,2023-03-31,"3,039,348.05951","16,548.00000",06254010


202203_202307 (342, 11)


,PERMCO,raw_cum_ret_202203_202307,ew_adj_cum_ret_202203_202307,vw_adj_cum_ret_202203_202307,max_drawdown_202203_202307,n_months_202203_202307,first_date_202203_202307,last_date_202203_202307,avg_mktcap_crsp_202203_202307,representative_permno_202203_202307,representative_cusip_202203_202307
0,90.00000,-0.11397,0.02122,-0.06518,-0.27369,17,2022-03-31,2023-07-31,"120,525,469.98835","59,176.00000",02581610
1,137.00000,0.02006,0.19627,0.07845,-0.23372,17,2022-03-31,2023-07-31,"42,985,751.91817","66,800.00000",02687478
2,362.00000,-0.18226,0.01048,-0.10134,-0.38389,17,2022-03-31,2023-07-31,"3,053,538.60805","15,318.00000",04548710
3,475.00000,-0.17062,-0.02035,-0.13299,-0.27443,17,2022-03-31,2023-07-31,"5,222,287.84131","16,505.00000",73317470
5,589.00000,-0.30074,-0.11445,-0.23228,-0.50791,17,2022-03-31,2023-07-31,"2,758,139.60003","16,548.00000",06254010


202303 (343, 11)


,PERMCO,raw_cum_ret_202303,ew_adj_cum_ret_202303,vw_adj_cum_ret_202303,max_drawdown_202303,n_months_202303,first_date_202303,last_date_202303,avg_mktcap_crsp_202303,representative_permno_202303,representative_cusip_202303
0,90.00000,-0.05196,0.12608,0.09247,0.00000,1,2023-03-31,2023-03-31,"122,754,633.07890","59,176.00000",02581610
1,137.00000,-0.17068,0.00736,-0.02625,0.00000,1,2023-03-31,2023-03-31,"37,127,759.36998","66,800.00000",02687478
2,362.00000,-0.22333,-0.04529,-0.07890,0.00000,1,2023-03-31,2023-03-31,"2,706,745.09109","15,318.00000",04548710
3,475.00000,-0.18824,-0.01020,-0.04381,0.00000,1,2023-03-31,2023-03-31,"4,131,568.04902","16,505.00000",73317470
4,589.00000,-0.30430,-0.12626,-0.15988,0.00000,1,2023-03-31,2023-03-31,"2,072,002.87285","16,548.00000",06254010


In [42]:
# =========================================================
# Cell 13. Compustat funda에서 2020-2021 회계자료 추출
# =========================================================

funda_cols = [
    "gvkey", "datadate", "fyear", "fyr",
    "indfmt", "consol", "popsrc", "datafmt",
    "tic", "cusip", "conm",
    "sich", "naicsh", "costat", "fic",
    "at", "ceq", "seq", "lt", "ni", "sale",
    "che", "dltt", "dlc", "prcc_f", "csho",
    "act", "lct"
]

def filter_funda_2020_2021_us_standard(chunk):
    chunk["fyear"] = pd.to_numeric(chunk["fyear"], errors="coerce")

    # 2020: asset growth 계산용
    # 2021: pre-shock accounting variables
    chunk = chunk[chunk["fyear"].isin([2020, 2021])].copy()

    # 미국 기업
    if "fic" in chunk.columns:
        chunk = chunk[chunk["fic"].eq("USA")].copy()

    # Compustat standard filters
    if "indfmt" in chunk.columns:
        chunk = chunk[chunk["indfmt"].eq("INDL")].copy()
    if "consol" in chunk.columns:
        chunk = chunk[chunk["consol"].eq("C")].copy()
    if "datafmt" in chunk.columns:
        chunk = chunk[chunk["datafmt"].eq("STD")].copy()

    return chunk


funda_raw = read_sas_filtered(
    file_paths["funda"],
    columns=funda_cols,
    chunksize=300_000,
    filter_func=filter_funda_2020_2021_us_standard
)

funda = funda_raw.copy()

# 숫자형 변환
num_cols = [
    "fyear", "at", "ceq", "seq", "lt", "ni", "sale",
    "che", "dltt", "dlc", "prcc_f", "csho",
    "act", "lct", "sich"
]

for col in num_cols:
    if col in funda.columns:
        funda[col] = pd.to_numeric(funda[col], errors="coerce")

funda["sich_num"] = pd.to_numeric(funda["sich"], errors="coerce")
funda["cusip8"] = funda["cusip"].astype(str).str[:8]

print("funda shape:", funda.shape)
print("Unique gvkey:", funda["gvkey"].nunique())
display(funda.head())

Reading chunk 1: (300000, 949)
Reading chunk 2: (300000, 949)
Reading chunk 3: (300000, 949)
Reading chunk 4: (38671, 949)
Final loaded shape: (16835, 28)
funda shape: (16835, 30)
Unique gvkey: 8969


,gvkey,datadate,fyear,fyr,indfmt,consol,popsrc,datafmt,tic,cusip,conm,sich,naicsh,costat,fic,at,ceq,seq,lt,ni,sale,che,dltt,dlc,prcc_f,csho,act,lct,sich_num,cusip8
0,001004,2021-05-31,"2,020.00000",5.00000,INDL,C,D,STD,AIR,000361105,AAR CORP,"5,080.00000","423,860.00000",A,USA,"1,539.70000",974.40000,974.40000,565.30000,35.80000,"1,651.40000",60.20000,193.60000,11.50000,41.75000,35.37500,937.00000,336.80000,"5,080.00000",00036110
1,001004,2022-05-31,"2,021.00000",5.00000,INDL,C,D,STD,AIR,000361105,AAR CORP,"5,080.00000","423,860.00000",A,USA,"1,573.90000","1,034.50000","1,034.50000",539.40000,78.70000,"1,817.10000",58.90000,156.30000,11.10000,48.22000,35.39100,"1,007.20000",348.20000,"5,080.00000",00036110
2,001019,2020-12-31,"2,020.00000",12.00000,INDL,C,D,STD,AFAP,001038108,AFA PROTECTIVE SYSTEMS INC,"7,380.00000","561,621.00000",I,USA,40.57000,13.47900,13.47900,27.09100,2.60300,82.67900,6.05300,6.66800,2.00000,174.00000,0.16200,31.09800,15.57300,"7,380.00000",00103810
3,001045,2020-12-31,"2,020.00000",12.00000,INDL,C,D,STD,AAL,02376R102,AMERICAN AIRLINES GROUP INC,"4,512.00000","481,111.00000",A,USA,"62,008.00000","-6,867.00000","-6,867.00000","68,875.00000","-8,885.00000","17,337.00000","7,473.00000","36,573.00000","4,448.00000",15.77000,621.48000,"11,095.00000","16,569.00000","4,512.00000",02376R10
4,001045,2021-12-31,"2,021.00000",12.00000,INDL,C,D,STD,AAL,02376R102,AMERICAN AIRLINES GROUP INC,"4,512.00000","481,111.00000",A,USA,"66,467.00000","-7,340.00000","-7,340.00000","73,807.00000","-1,993.00000","29,882.00000","13,421.00000","42,181.00000","3,996.00000",17.96000,647.72800,"17,336.00000","19,006.00000","4,512.00000",02376R10


In [43]:
# =========================================================
# Cell 14. 2021년 pre-shock 회계변수 생성
# =========================================================

# 2020 총자산: asset growth 계산용
at_2020 = (
    funda[funda["fyear"] == 2020]
    [["gvkey", "at"]]
    .dropna(subset=["gvkey", "at"])
    .sort_values(["gvkey", "at"], ascending=[True, False])
    .drop_duplicates(subset=["gvkey"])
    .rename(columns={"at": "at_2020"})
)

# 2021 회계정보
x_2021 = funda[funda["fyear"] == 2021].copy()

# market cap 계산
x_2021["market_cap_compustat"] = x_2021["prcc_f"] * x_2021["csho"]

# 중복 제거
x_2021 = (
    x_2021
    .sort_values(
        ["gvkey", "cusip8", "market_cap_compustat"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["gvkey", "cusip8"])
    .copy()
)

# 2020 자산 붙이기
x_2021 = x_2021.merge(at_2020, on="gvkey", how="left")

# 주요 변수 생성
x_2021["size"] = np.log(x_2021["at"])
x_2021["equity_ratio"] = safe_divide(x_2021["ceq"], x_2021["at"])
x_2021["roa"] = safe_divide(x_2021["ni"], x_2021["at"])
x_2021["cash_ratio"] = safe_divide(x_2021["che"], x_2021["at"])
x_2021["debt_ratio"] = safe_divide(x_2021["dltt"].fillna(0) + x_2021["dlc"].fillna(0), x_2021["at"])
x_2021["market_to_book"] = safe_divide(x_2021["market_cap_compustat"], x_2021["ceq"])
x_2021["asset_growth"] = safe_divide(x_2021["at"] - x_2021["at_2020"], x_2021["at_2020"])

# 참고용 변수
x_2021["leverage"] = safe_divide(x_2021["lt"], x_2021["at"])
x_2021["current_ratio"] = safe_divide(x_2021["act"], x_2021["lct"])

x_vars_main = [
    "size",
    "equity_ratio",
    "roa",
    "cash_ratio",
    "debt_ratio",
    "asset_growth",
    "market_to_book"
]

x_vars_extended = x_vars_main + [
    "leverage",
    "current_ratio"
]

x_data_all = x_2021[
    [
        "gvkey", "cusip", "cusip8", "tic", "conm",
        "sich_num", "naicsh", "at", "market_cap_compustat"
    ] + x_vars_extended
].copy()

x_data_all = x_data_all.replace([np.inf, -np.inf], np.nan)

print("x_data_all shape:", x_data_all.shape)
print("Unique gvkey:", x_data_all["gvkey"].nunique())
display(x_data_all.head())
display(x_data_all[x_vars_extended].describe().T)

x_data_all shape: (8555, 18)
Unique gvkey: 8555


,gvkey,cusip,cusip8,tic,conm,sich_num,naicsh,at,market_cap_compustat,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book,leverage,current_ratio
0,001004,000361105,00036110,AIR,AAR CORP,"5,080.00000","423,860.00000","1,573.90000","1,706.55402",7.36131,0.65728,0.05000,0.03742,0.10636,0.02221,1.64964,0.34272,2.89259
1,001045,02376R102,02376R10,AAL,AMERICAN AIRLINES GROUP INC,"4,512.00000","481,111.00000","66,467.00000","11,633.19488",11.10446,-0.11043,-0.02998,0.20192,0.69474,0.07191,-1.58490,1.11043,0.91213
2,001050,125141101,12514110,CECO,CECO ENVIRONMENTAL CORP,"3,564.00000","333,413.00000",416.19700,218.22444,6.03116,0.49148,0.00343,0.07687,0.19639,-0.00743,1.06683,0.50515,1.61984
3,001075,723484101,72348410,PNW,PINNACLE WEST CAPITAL CORP,"4,911.00000","2,211.00000","22,003.22200","7,971.51693",9.99894,0.26842,0.02812,0.00045,0.37197,0.09904,1.34969,0.72634,0.88288
4,001076,74319R101,74319R10,PRG,PROG HOLDINGS INC,"6,141.00000","522,220.00000","1,621.76100","2,546.05351",7.39127,0.41893,0.15018,0.10492,0.37926,0.23103,3.74746,0.58107,NaN


,count,mean,std,min,25%,50%,75%,max
size,"5,369.00000",6.39435,2.87502,-6.90776,4.78733,6.73497,8.27825,15.25752
equity_ratio,"5,358.00000",-1.93289,37.37351,"-1,427.66176",0.13423,0.38896,0.65289,1.53920
roa,"5,358.00000",-4.47345,215.52545,"-15,612.60000",-0.16602,0.00936,0.05162,20.20000
cash_ratio,"5,369.00000",0.28133,0.30306,0.00000,0.04634,0.14083,0.45549,1.00000
debt_ratio,"5,369.00000",1.16501,15.34625,0.00000,0.03874,0.21299,0.42615,629.00000
asset_growth,"5,057.00000",55.95906,"2,335.85680",-1.00000,-0.00299,0.09157,0.33467,"150,021.00000"
market_to_book,"5,038.00000",31.01920,"1,404.14163","-11,109.50000",1.05571,2.00730,4.40503,"82,826.02632"
leverage,"5,359.00000",2.82930,36.71149,0.00000,0.32624,0.58117,0.83488,"1,399.25000"
current_ratio,"4,268.00000",6.78864,124.12340,0.00000,1.13135,1.99455,4.35145,"8,074.41667"


In [44]:
# =========================================================
# Cell 15. PERMCO 은행 표본과 Compustat 회계변수 연결
# =========================================================

# CRSP에서 관측된 PERMCO-CUSIP mapping 생성
permco_cusip_map = (
    permco_monthly
    .dropna(subset=["PERMCO", "CUSIP"])
    .groupby(["PERMCO", "CUSIP"], as_index=False)
    .agg(
        n_months_map=("DATE", "nunique"),
        avg_mktcap_map=("mktcap_crsp", "mean"),
        representative_permno_map=("PERMNO", "first")
    )
)

print("permco_cusip_map shape:", permco_cusip_map.shape)
display(permco_cusip_map.head())

# CUSIP으로 Compustat 연결
permco_x_candidates = permco_cusip_map.merge(
    x_data_all,
    left_on="CUSIP",
    right_on="cusip8",
    how="inner"
)

print("PERMCO-Compustat candidate matches:", permco_x_candidates.shape)
print("Matched unique PERMCO:", permco_x_candidates["PERMCO"].nunique())
print("Matched unique gvkey:", permco_x_candidates["gvkey"].nunique())

# PERMCO별 대표 Compustat record 선택
# 우선순위: mapping 관측월 수 많음, CRSP 평균시총 큼, Compustat 시총 큼
permco_x = (
    permco_x_candidates
    .sort_values(
        ["PERMCO", "n_months_map", "avg_mktcap_map", "market_cap_compustat"],
        ascending=[True, False, False, False]
    )
    .drop_duplicates(subset=["PERMCO"], keep="first")
    .copy()
)

print("Final PERMCO accounting data:", permco_x.shape)
print("Unique PERMCO:", permco_x["PERMCO"].nunique())
print("Unique gvkey:", permco_x["gvkey"].nunique())

# 매칭률 진단
matched_permcos = set(permco_x["PERMCO"].dropna().astype(int))
return_permcos = set(y_main["PERMCO"].dropna().astype(int))
unmatched_permcos = sorted(list(return_permcos - matched_permcos))

print("PERMCOs with return data:", len(return_permcos))
print("PERMCOs matched to Compustat:", len(matched_permcos))
print("Return PERMCOs not matched to Compustat:", len(unmatched_permcos))

display(permco_x[["PERMCO", "gvkey", "conm", "tic", "sich_num", "CUSIP", "cusip8"] + x_vars_main].head(50))

permco_cusip_map shape: (362, 5)


,PERMCO,CUSIP,n_months_map,avg_mktcap_map,representative_permno_map
0,90.00000,02581610,17,"120,525,469.98835","59,176.00000"
1,137.00000,02687478,17,"42,985,751.91817","66,800.00000"
2,362.00000,04548710,17,"3,053,538.60805","15,318.00000"
3,475.00000,73317470,17,"5,222,287.84131","16,505.00000"
4,579.00000,65252620,2,"490,877.63122","58,836.00000"


PERMCO-Compustat candidate matches: (354, 23)
Matched unique PERMCO: 354
Matched unique gvkey: 354
Final PERMCO accounting data: (354, 23)
Unique PERMCO: 354
Unique gvkey: 354
PERMCOs with return data: 345
PERMCOs matched to Compustat: 354
Return PERMCOs not matched to Compustat: 7


,PERMCO,gvkey,conm,tic,sich_num,CUSIP,cusip8,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
0,90.00000,001447,AMERICAN EXPRESS CO,AXP,"6,141.00000",02581610,02581610,12.14711,0.11762,0.04275,0.12466,0.22296,-0.01473,5.61391
1,137.00000,001487,AMERICAN INTERNATIONAL GROUP,AIG,"6,331.00000",02687478,02687478,13.29818,0.10983,0.01575,0.03895,0.06484,0.01642,0.71101
2,362.00000,011842,ASSOCIATED BANC-CORP,ASB,"6,020.00000",04548710,04548710,10.46608,0.10915,0.01000,0.02921,0.06427,0.05040,0.88047
3,475.00000,002002,POPULAR INC,BPOP,"6,020.00000",73317470,73317470,11.22655,0.07919,0.01245,0.23922,0.01770,0.13912,1.10151
4,579.00000,008060,NEWTEKONE INC,NEWT,"6,797.00000",65252620,65252620,6.96277,0.38227,0.07964,0.17686,0.45535,0.25635,1.65272
5,589.00000,002005,BANK OF HAWAII CORP,BOH,"6,020.00000",06254010,06254010,10.03386,0.06283,0.01112,0.02460,0.02476,0.10587,2.35510
6,779.00000,003238,COMMERCE BANCSHARES INC,CBSH,"6,020.00000",20052510,20052510,10.51023,0.09369,0.01447,0.11664,0.08348,0.11439,2.43347
7,781.00000,013041,SYNOVUS FINANCIAL CORP,SNV,"6,020.00000",87161C50,87161C50,10.95636,0.08304,0.01327,0.05251,0.02562,0.05374,1.45843
8,840.00000,003643,CULLEN/FROST BANKERS INC,CFR,"6,020.00000",22989910,22989910,10.83720,0.08440,0.00871,0.32593,0.06491,0.20021,1.87856
9,"1,261.00000",003231,COMERICA INC,CMA,"6,020.00000",20034010,20034010,11.45758,0.07930,0.01234,0.23970,0.03331,0.07361,1.51537


In [45]:
# =========================================================
# Cell 16. 최종 분석 데이터 생성
# =========================================================

analysis_df = y_main.merge(
    permco_x,
    on="PERMCO",
    how="left"
)

analysis_df = analysis_df.merge(
    y_extended.drop(columns=[
        "representative_permno_202203_202307",
        "representative_cusip_202203_202307"
    ], errors="ignore"),
    on="PERMCO",
    how="left"
)

analysis_df = analysis_df.merge(
    y_march.drop(columns=[
        "representative_permno_202303",
        "representative_cusip_202303"
    ], errors="ignore"),
    on="PERMCO",
    how="left"
)

# CRSP PERMCO 회사정보 붙이기
if len(crsp_permco_info) > 0:
    analysis_df = analysis_df.merge(
        crsp_permco_info,
        on="PERMCO",
        how="left",
        suffixes=("", "_crsp")
    )

print("analysis_df shape:", analysis_df.shape)
print("Unique PERMCO:", analysis_df["PERMCO"].nunique())
print("Unique gvkey matched:", analysis_df["gvkey"].nunique())

print("\nCompustat match missing:")
print(analysis_df["gvkey"].isna().sum())

print("\nSIC distribution from Compustat:")
display(analysis_df["sich_num"].value_counts(dropna=False).sort_index())

display(
    analysis_df[
        [
            "PERMCO", "gvkey", "conm", "tic", "COMNAM", "TICKER",
            "sich_num", "representative_permno_202203_202303",
            "raw_cum_ret_202203_202303",
            "vw_adj_cum_ret_202203_202303"
        ] + x_vars_main
    ].head(50)
)

analysis_df shape: (345, 55)
Unique PERMCO: 345
Unique gvkey matched: 338

Compustat match missing:
7

SIC distribution from Compustat:


sich_num
6,020.00000    280
6,035.00000     21
6,036.00000     14
6,141.00000      6
6,162.00000      1
6,172.00000      1
6,199.00000      2
6,211.00000      4
6,282.00000      5
6,311.00000      1
6,331.00000      1
6,361.00000      1
7,389.00000      1
NaN              7
Name: count, dtype: int64

,PERMCO,gvkey,conm,tic,COMNAM,TICKER,sich_num,representative_permno_202203_202303,raw_cum_ret_202203_202303,vw_adj_cum_ret_202203_202303,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
0,90.00000,001447,AMERICAN EXPRESS CO,AXP,AMERICAN EXPRESS CO,AXP,"6,141.00000","59,176.00000",-0.14085,0.05586,12.14711,0.11762,0.04275,0.12466,0.22296,-0.01473,5.61391
1,137.00000,001487,AMERICAN INTERNATIONAL GROUP,AIG,AMERICAN INTERNATIONAL GROUP INC,AIG,"6,331.00000","66,800.00000",-0.15311,0.03360,13.29818,0.10983,0.01575,0.03895,0.06484,0.01642,0.71101
2,362.00000,011842,ASSOCIATED BANC-CORP,ASB,ASSOCIATED BANC CORP,ASB,"6,020.00000","15,318.00000",-0.23497,-0.04623,10.46608,0.10915,0.01000,0.02921,0.06427,0.05040,0.88047
3,475.00000,002002,POPULAR INC,BPOP,POPULAR INC,BPOP,"6,020.00000","16,505.00000",-0.34995,-0.22550,11.22655,0.07919,0.01245,0.23922,0.01770,0.13912,1.10151
4,589.00000,002005,BANK OF HAWAII CORP,BOH,BANK OF HAWAII CORP,BOH,"6,020.00000","16,548.00000",-0.37375,-0.22974,10.03386,0.06283,0.01112,0.02460,0.02476,0.10587,2.35510
5,779.00000,003238,COMMERCE BANCSHARES INC,CBSH,COMMERCE BANCSHARES INC,CBSH,"6,020.00000","25,129.00000",-0.12918,0.02481,10.51023,0.09369,0.01447,0.11664,0.08348,0.11439,2.43347
6,781.00000,013041,SYNOVUS FINANCIAL CORP,SNV,SYNOVUS FINANCIAL CORP,SNV,"6,020.00000","20,053.00000",-0.38651,-0.21985,10.95636,0.08304,0.01327,0.05251,0.02562,0.05374,1.45843
7,840.00000,003643,CULLEN/FROST BANKERS INC,CFR,CULLEN FROST BANKERS INC,CFR,"6,020.00000","27,888.00000",-0.23236,-0.07646,10.83720,0.08440,0.00871,0.32593,0.06491,0.20021,1.87856
8,"1,261.00000",003231,COMERICA INC,CMA,COMERICA INC,CMA,"6,020.00000","25,081.00000",-0.52075,-0.39178,11.45758,0.07930,0.01234,0.23970,0.03331,0.07361,1.51537
9,"1,620.00000",004674,REGIONS FINANCIAL CORP,RF,REGIONS FINANCIAL CORP NEW,RF,"6,020.00000","35,044.00000",-0.19622,0.00294,12.00113,0.10229,0.01547,0.18050,0.01802,0.10550,1.23199


In [46]:
# =========================================================
# Cell 17. 회귀분석용 데이터 정리
# =========================================================

main_y = "vw_adj_cum_ret_202203_202303"

y_vars = [
    "raw_cum_ret_202203_202303",
    "ew_adj_cum_ret_202203_202303",
    "vw_adj_cum_ret_202203_202303",
    "max_drawdown_202203_202303",
    "raw_cum_ret_202203_202307",
    "ew_adj_cum_ret_202203_202307",
    "vw_adj_cum_ret_202203_202307",
    "max_drawdown_202203_202307",
    "raw_cum_ret_202303",
    "ew_adj_cum_ret_202303",
    "vw_adj_cum_ret_202303",
    "max_drawdown_202303"
]

id_cols = [
    "PERMCO", "gvkey", "conm", "tic", "COMNAM", "TICKER",
    "cusip8", "representative_cusip_202203_202303",
    "representative_permno_202203_202303",
    "sich_num", "naicsh", "SICCD", "NAICS"
]

existing_id_cols = [c for c in id_cols if c in analysis_df.columns]
existing_y_vars = [c for c in y_vars if c in analysis_df.columns]

reg_cols = existing_id_cols + existing_y_vars + x_vars_extended

reg_df = analysis_df[reg_cols].copy()
reg_df = reg_df.replace([np.inf, -np.inf], np.nan)

# 숫자형 변환
for col in existing_y_vars + x_vars_extended:
    if col in reg_df.columns:
        reg_df[col] = pd.to_numeric(reg_df[col], errors="coerce")

# winsorize
for col in existing_y_vars + x_vars_extended:
    if col in reg_df.columns:
        reg_df[col] = winsorize_series(reg_df[col], 0.01, 0.99)

# 메인 회귀 표본
reg_main = reg_df.dropna(subset=[main_y] + x_vars_main).copy()

print("reg_df shape:", reg_df.shape)
print("reg_main shape:", reg_main.shape)
print("Unique PERMCO in reg_main:", reg_main["PERMCO"].nunique())
print("Unique gvkey in reg_main:", reg_main["gvkey"].nunique())

print("\nMissing counts:")
display(reg_df[[main_y] + x_vars_main].isna().sum())

print("\nSummary statistics:")
display(reg_main[[main_y] + x_vars_main].describe().T)

print("\nCorrelation matrix:")
display(reg_main[[main_y] + x_vars_main].corr())

reg_df shape: (345, 34)
reg_main shape: (336, 34)
Unique PERMCO in reg_main: 336
Unique gvkey in reg_main: 336

Missing counts:


vw_adj_cum_ret_202203_202303    0
size                            7
equity_ratio                    7
roa                             7
cash_ratio                      7
debt_ratio                      7
asset_growth                    8
market_to_book                  8
dtype: int64


Summary statistics:


,count,mean,std,min,25%,50%,75%,max
vw_adj_cum_ret_202203_202303,336.00000,-0.05043,0.20415,-0.75413,-0.18053,-0.04839,0.07535,0.40138
size,336.00000,8.91627,1.66781,6.24381,7.69725,8.63320,9.73148,14.37665
equity_ratio,336.00000,0.10723,0.03941,0.05870,0.08827,0.10105,0.11702,0.42227
roa,336.00000,0.01264,0.00784,-0.00075,0.00937,0.01159,0.01413,0.06581
cash_ratio,336.00000,0.10954,0.07982,0.00895,0.04692,0.09014,0.14841,0.35697
debt_ratio,336.00000,0.05173,0.05347,0.00050,0.02198,0.03596,0.06127,0.34349
asset_growth,336.00000,0.15026,0.19514,-0.06924,0.04761,0.09871,0.17606,1.18327
market_to_book,336.00000,1.38901,0.65511,0.68879,1.05071,1.21058,1.45542,4.82887



Correlation matrix:


,vw_adj_cum_ret_202203_202303,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
vw_adj_cum_ret_202203_202303,1.00000,-0.12309,0.12327,0.10357,0.20304,-0.06176,-0.13692,-0.14559
size,-0.12309,1.00000,-0.01509,0.16604,0.12573,0.28974,-0.05044,0.22622
equity_ratio,0.12327,-0.01509,1.00000,0.70686,0.00683,0.14765,-0.02343,0.18091
roa,0.10357,0.16604,0.70686,1.00000,0.05156,0.22523,-0.16153,0.41178
cash_ratio,0.20304,0.12573,0.00683,0.05156,1.00000,-0.01698,0.23500,0.14670
debt_ratio,-0.06176,0.28974,0.14765,0.22523,-0.01698,1.00000,-0.12227,0.11494
asset_growth,-0.13692,-0.05044,-0.02343,-0.16153,0.23500,-0.12227,1.00000,0.14728
market_to_book,-0.14559,0.22622,0.18091,0.41178,0.14670,0.11494,0.14728,1.00000


In [47]:
# =========================================================
# Cell 18. 표본 진단표 생성
# =========================================================

sample_diagnostics = pd.DataFrame([
    {
        "step": "Raw CRSP-FRB Link file",
        "N_PERMCO": bank_permco_link_all["PERMCO"].nunique(),
        "N_rows": len(bank_permco_link_all)
    },
    {
        "step": "CRSP-FRB links active during 2021-2023",
        "N_PERMCO": bank_permco_link_filtered["PERMCO"].nunique(),
        "N_rows": len(bank_permco_link_filtered)
    },
    {
        "step": "One row per active PERMCO bank list",
        "N_PERMCO": len(bank_permcos),
        "N_rows": len(bank_permco_list_raw)
    },
    {
        "step": "CRSP bank PERMCOs with monthly return data",
        "N_PERMCO": permco_monthly["PERMCO"].nunique(),
        "N_rows": len(permco_monthly)
    },
    {
        "step": "PERMCOs with main-period Y variable",
        "N_PERMCO": y_main["PERMCO"].nunique(),
        "N_rows": len(y_main)
    },
    {
        "step": "PERMCOs matched to Compustat 2021 accounting data",
        "N_PERMCO": permco_x["PERMCO"].nunique(),
        "N_rows": len(permco_x)
    },
    {
        "step": "Final main regression sample",
        "N_PERMCO": reg_main["PERMCO"].nunique(),
        "N_rows": len(reg_main)
    }
])

print("Sample diagnostics:")
display(sample_diagnostics)

print("\nInstitution type distribution in active CRSP-FRB links:")
inst_type_active = (
    bank_permco_link_filtered["inst_type"]
    .value_counts(dropna=False)
    .reset_index()
)
inst_type_active.columns = ["inst_type", "N"]
display(inst_type_active)

print("\nSIC distribution in final regression sample:")
if "sich_num" in reg_main.columns:
    sic_distribution = reg_main["sich_num"].value_counts(dropna=False).sort_index().reset_index()
    sic_distribution.columns = ["sich_num", "N"]
    display(sic_distribution)

print("\nFinal sample firms, first 30 by dependent variable:")
display(
    reg_main[
        ["PERMCO", "gvkey", "conm", "tic", "COMNAM", "TICKER", "sich_num", main_y] + x_vars_main
    ].sort_values(main_y).head(30)
)

print("\nTop performers:")
display(
    reg_main[
        ["PERMCO", "gvkey", "conm", "tic", "COMNAM", "TICKER", "sich_num", main_y, "raw_cum_ret_202203_202303"] + x_vars_main
    ].sort_values(main_y, ascending=False).head(20)
)

print("\nWorst performers:")
display(
    reg_main[
        ["PERMCO", "gvkey", "conm", "tic", "COMNAM", "TICKER", "sich_num", main_y, "raw_cum_ret_202203_202303"] + x_vars_main
    ].sort_values(main_y, ascending=True).head(20)
)

Sample diagnostics:


,step,N_PERMCO,N_rows
0,Raw CRSP-FRB Link file,1450,1495
1,CRSP-FRB links active during 2021-2023,394,395
2,One row per active PERMCO bank list,394,394
3,CRSP bank PERMCOs with monthly return data,362,5898
4,PERMCOs with main-period Y variable,345,345
5,PERMCOs matched to Compustat 2021 accounting data,354,354
6,Final main regression sample,336,336



Institution type distribution in active CRSP-FRB links:


,inst_type,N
0,Bank Holding Company,320
1,Thrift Holding Company,32
2,Thrift holding company,24
3,Commercial Bank,6
4,Non Member Bank,5
5,Domestic Entity Other,2
6,Bank holding company,2
7,NaN,1
8,Thirft Holding Company,1
9,FSB,1



SIC distribution in final regression sample:


...

,PERMCO,gvkey,conm,tic,COMNAM,TICKER,sich_num,vw_adj_cum_ret_202203_202303,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
199,"44,999.00000",160776,SIGNATURE BANK/NY,SBNY,SIGNATURE BANK NEW YORK N Y,SBNY,"6,020.00000",-0.75413,11.68221,0.06620,0.00775,0.25008,0.03051,0.60303,2.50142
243,"53,609.00000",014275,FIRST REPUBLIC BANK,FRCB,FIRST REPUBLIC BANK S F NEW,FRC,"6,020.00000",-0.75413,12.10673,0.06773,0.00816,0.07150,0.03800,0.27077,3.02185
331,"56,932.00000",034446,SILVERGATE CAPITAL CORP,SICP,SILVERGATE CAPITAL CORP,SI,"6,020.00000",-0.75413,9.68069,0.10052,0.00491,0.33663,0.00132,1.18327,2.80061
66,"9,999.00000",018040,REPUBLIC FIRST BANCORP INC,FRBKQ,REPUBLIC FIRST BANCORP INC,FRBK,"6,020.00000",-0.67606,8.63527,0.05870,0.00447,0.02113,0.01654,0.11073,0.68879
272,"55,109.00000",020677,FIRST FOUNDATION INC,FFWM,FIRST FOUNDATION INC,FFWM,"6,020.00000",-0.63441,9.22977,0.10436,0.01074,0.11002,0.02263,0.46557,1.31845
259,"54,408.00000",170396,CUSTOMERS BANCORP INC,CUBI,CUSTOMERS BANCORP INC,CUBI,"6,020.00000",-0.59259,9.88201,0.06275,0.01607,0.02646,0.06101,0.06160,1.75145
257,"54,332.00000",162293,FIRST INTERNET BANCORP,INBK,FIRST INTERNET BANCORP,INBK,"6,020.00000",-0.58951,8.34545,0.09032,0.01143,0.10519,0.14703,-0.00828,1.20637
296,"56,141.00000",032519,METROPOLITAN BANK HLDNG,MCB,METROPOLITAN BANK HOLDING CORP,MCB,"6,020.00000",-0.58492,8.87015,0.07827,0.00851,0.33154,0.01093,0.64319,2.08876
252,"54,001.00000",018948,MECHANICS BANCORP,MCHB,HOMESTREET INC,HMST,"6,020.00000",-0.54871,9.87421,0.12294,0.01061,0.05155,0.00324,0.01462,0.75283
274,"55,129.00000",019647,LENDINGCLUB CORP,LC,LENDINGCLUB CORP,LC,"6,141.00000",-0.50554,8.49706,0.17351,0.00379,0.15582,0.14324,1.18327,2.87359



Top performers:


,PERMCO,gvkey,conm,tic,COMNAM,TICKER,sich_num,vw_adj_cum_ret_202203_202303,raw_cum_ret_202203_202303,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
290,"55,987.00000",031606,ESQUIRE FINCL HOLD INC,ESQ,ESQUIRE FINANCIAL HOLDINGS,ESQ,"6,020.00000",0.40138,0.16918,7.07223,0.12194,0.01521,0.16918,0.00253,0.25841,1.77273
50,"8,674.00000",004690,FIRST CITIZENS BANCSH -CL A,FCNCA,FIRST CITIZENS BANCSHARES INC NC,FCNCA,"6,020.00000",0.40138,0.22487,10.97350,0.07541,0.00939,0.16211,0.03169,0.16715,1.85243
236,"53,116.00000",066158,OAK VALLEY BANCORP,OVLY,OAK VALLEY BANCORP,OVLY,"6,020.00000",0.40138,0.22487,7.58298,0.07260,0.00832,0.35697,0.00380,0.29971,1.00524
222,"51,209.00000",174301,LIMESTONE BANCORP INC,LMST,LIMESTONE BANCORP INC,LMST,"6,020.00000",0.40138,0.20798,7.25537,0.09251,0.01053,0.05482,0.05038,0.07879,1.08219
61,"9,280.00000",016781,CITY HOLDING CO,CHCO,CITY HOLDING CO,CHCO,"6,020.00000",0.38391,0.17295,8.70013,0.11345,0.01467,0.10571,0.05204,0.04255,1.80871
334,"57,031.00000",023552,PROFESSIONAL HOLDING CORP,PFHD,PROFESSIONAL HOLDING CORP,PFHD,"6,020.00000",0.37020,0.22487,7.88763,0.08690,0.00802,0.22427,0.01882,0.29497,1.11274
143,"17,665.00000",107325,MACATAWA BANK CORP,MCBC,MACATAWA BANK CORP,MCBC,"6,020.00000",0.35600,0.14885,7.98233,0.08673,0.00991,0.35697,0.02932,0.10852,1.18963
139,"16,640.00000",117026,GREENE COUNTY BANCORP INC,GCBC,GREENE COUNTY BANCORP INC,GCBC,"6,035.00000",0.34865,0.12349,7.69636,0.06798,0.01088,0.07014,0.01116,0.31222,1.60034
109,"14,781.00000",025201,CASS INFORMATION SYSTEMS INC,CASS,CASS INFORMATION SYSTEMS INC,CASS,"7,389.00000",0.33257,0.13529,7.84577,0.09621,0.01120,0.21102,0.00191,0.15961,2.19701
44,"7,788.00000",016989,PEOPLES BANCORP NC INC,PEBK,PEOPLES BANCORP NC INC NEW,PEBK,"6,020.00000",0.32665,0.15601,7.39277,0.08766,0.00932,0.17085,0.03524,0.14796,1.09725



Worst performers:


,PERMCO,gvkey,conm,tic,COMNAM,TICKER,sich_num,vw_adj_cum_ret_202203_202303,raw_cum_ret_202203_202303,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
199,"44,999.00000",160776,SIGNATURE BANK/NY,SBNY,SIGNATURE BANK NEW YORK N Y,SBNY,"6,020.00000",-0.75413,-0.86465,11.68221,0.06620,0.00775,0.25008,0.03051,0.60303,2.50142
243,"53,609.00000",014275,FIRST REPUBLIC BANK,FRCB,FIRST REPUBLIC BANK S F NEW,FRC,"6,020.00000",-0.75413,-0.86465,12.10673,0.06773,0.00816,0.07150,0.03800,0.27077,3.02185
331,"56,932.00000",034446,SILVERGATE CAPITAL CORP,SICP,SILVERGATE CAPITAL CORP,SI,"6,020.00000",-0.75413,-0.86465,9.68069,0.10052,0.00491,0.33663,0.00132,1.18327,2.80061
66,"9,999.00000",018040,REPUBLIC FIRST BANCORP INC,FRBKQ,REPUBLIC FIRST BANCORP INC,FRBK,"6,020.00000",-0.67606,-0.73796,8.63527,0.05870,0.00447,0.02113,0.01654,0.11073,0.68879
272,"55,109.00000",020677,FIRST FOUNDATION INC,FFWM,FIRST FOUNDATION INC,FFWM,"6,020.00000",-0.63441,-0.71338,9.22977,0.10436,0.01074,0.11002,0.02263,0.46557,1.31845
259,"54,408.00000",170396,CUSTOMERS BANCORP INC,CUBI,CUSTOMERS BANCORP INC,CUBI,"6,020.00000",-0.59259,-0.69911,9.88201,0.06275,0.01607,0.02646,0.06101,0.06160,1.75145
257,"54,332.00000",162293,FIRST INTERNET BANCORP,INBK,FIRST INTERNET BANCORP,INBK,"6,020.00000",-0.58951,-0.65565,8.34545,0.09032,0.01143,0.10519,0.14703,-0.00828,1.20637
296,"56,141.00000",032519,METROPOLITAN BANK HLDNG,MCB,METROPOLITAN BANK HOLDING CORP,MCB,"6,020.00000",-0.58492,-0.66856,8.87015,0.07827,0.00851,0.33154,0.01093,0.64319,2.08876
252,"54,001.00000",018948,MECHANICS BANCORP,MCHB,HOMESTREET INC,HMST,"6,020.00000",-0.54871,-0.63426,9.87421,0.12294,0.01061,0.05155,0.00324,0.01462,0.75283
274,"55,129.00000",019647,LENDINGCLUB CORP,LC,LENDINGCLUB CORP,LC,"6,141.00000",-0.50554,-0.61195,8.49706,0.17351,0.00379,0.15582,0.14324,1.18327,2.87359


In [48]:
# =========================================================
# Cell 19. OLS 회귀분석 함수
# =========================================================

def run_ols(df, y_col, x_cols, model_name):
    temp = df.dropna(subset=[y_col] + x_cols).copy()

    y = temp[y_col]
    X = temp[x_cols]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit(cov_type="HC3")

    table = pd.DataFrame({
        "model": model_name,
        "y": y_col,
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "t_stat": model.tvalues.values,
        "p_value": model.pvalues.values
    })

    table["stars"] = table["p_value"].apply(make_stars)
    table["coef_stars"] = table["coef"].map(lambda x: f"{x:.4f}") + table["stars"]
    table["t_line"] = table["t_stat"].map(lambda x: f"({x:.2f})")

    fit = {
        "model": model_name,
        "y": y_col,
        "N": int(model.nobs),
        "R2": model.rsquared,
        "Adj_R2": model.rsquared_adj,
        "F_pvalue": model.f_pvalue
    }

    return model, table, fit


def make_regression_wide_table(ols_tables, fit_df=None):
    temp = ols_tables.copy()

    coef_wide = temp.pivot(index="variable", columns="model", values="coef_stars")
    t_wide = temp.pivot(index="variable", columns="model", values="t_line")

    rows = []
    for var in coef_wide.index:
        rows.append(pd.DataFrame(coef_wide.loc[[var]]))
        t_row = pd.DataFrame(t_wide.loc[[var]])
        t_row.index = [var + "_t"]
        rows.append(t_row)

    out = pd.concat(rows)

    if fit_df is not None and len(fit_df) > 0:
        for stat in ["N", "R2", "Adj_R2"]:
            stat_row = {}
            for _, row in fit_df.iterrows():
                m = row["model"]
                if stat == "N":
                    stat_row[m] = f"{int(row[stat])}"
                else:
                    stat_row[m] = f"{row[stat]:.4f}"
            out.loc[stat] = pd.Series(stat_row)

    return out


print("OLS functions ready.")

OLS functions ready.


In [49]:
# =========================================================
# Cell 20. Main Regression
# =========================================================

x_model1 = ["size", "equity_ratio", "roa"]

x_model2 = [
    "size",
    "equity_ratio",
    "roa",
    "cash_ratio",
    "debt_ratio",
    "asset_growth"
]

x_model3 = [
    "size",
    "equity_ratio",
    "roa",
    "cash_ratio",
    "debt_ratio",
    "asset_growth",
    "market_to_book"
]

ols1, ols_table1, fit1 = run_ols(reg_main, main_y, x_model1, "M1_basic")
ols2, ols_table2, fit2 = run_ols(reg_main, main_y, x_model2, "M2_balance_sheet")
ols3, ols_table3, fit3 = run_ols(reg_main, main_y, x_model3, "M3_full")

print("\n===== M1_basic =====")
print(ols1.summary())

print("\n===== M2_balance_sheet =====")
print(ols2.summary())

print("\n===== M3_full =====")
print(ols3.summary())

ols_tables = pd.concat([ols_table1, ols_table2, ols_table3], ignore_index=True)
ols_fit = pd.DataFrame([fit1, fit2, fit3])

main_regression_table_wide = make_regression_wide_table(ols_tables, ols_fit)

print("\nMain regression table, wide format:")
display(main_regression_table_wide)

display(ols_fit)


===== M1_basic =====
                                 OLS Regression Results                                 
Dep. Variable:     vw_adj_cum_ret_202203_202303   R-squared:                       0.033
Model:                                      OLS   Adj. R-squared:                  0.024
Method:                           Least Squares   F-statistic:                     4.236
Date:                          Wed, 27 May 2026   Prob (F-statistic):            0.00589
Time:                                  17:09:29   Log-Likelihood:                 63.238
No. Observations:                           336   AIC:                            -118.5
Df Residuals:                               332   BIC:                            -103.2
Df Model:                                     3                                         
Covariance Type:                            HC3                                         
                   coef    std err          z      P>|z|      [0.025      0.975]
-------

model,M1_basic,M2_balance_sheet,M3_full
asset_growth,NaN,-0.2197**,-0.1731**
asset_growth_t,NaN,(-2.41),(-2.05)
cash_ratio,NaN,0.6876***,0.7108***
cash_ratio_t,NaN,(4.52),(4.87)
const,0.0353,0.0164,0.0496
const_t,(0.50),(0.23),(0.70)
debt_ratio,NaN,-0.2252,-0.2189
debt_ratio_t,NaN,(-0.81),(-0.86)
equity_ratio,0.3324,0.5837,0.3607
equity_ratio_t,(0.83),(1.54),(1.06)


,model,y,N,R2,Adj_R2,F_pvalue
0,M1_basic,vw_adj_cum_ret_202203_202303,336,0.03295,0.02421,0.00589
1,M2_balance_sheet,vw_adj_cum_ret_202203_202303,336,0.12097,0.10494,0.00000
2,M3_full,vw_adj_cum_ret_202203_202303,336,0.14766,0.12947,0.00000


In [50]:
# =========================================================
# Cell 21. Robustness Tests
# =========================================================

robust_specs = [
    ("R1_raw_return_main_period", "raw_cum_ret_202203_202303", x_model3),
    ("R2_ew_bank_adjusted_main_period", "ew_adj_cum_ret_202203_202303", x_model3),
    ("R3_extended_vw_adjusted_202203_202307", "vw_adj_cum_ret_202203_202307", x_model3),
    ("R4_march_2023_vw_adjusted", "vw_adj_cum_ret_202303", x_model3),
    ("R5_max_drawdown_main_period", "max_drawdown_202203_202303", x_model3),
]

robust_models = {}
robust_tables_list = []
robust_fits_list = []

for model_name, y_col, x_cols in robust_specs:
    if y_col not in reg_df.columns:
        print(f"{model_name}: missing y variable {y_col}. Skipped.")
        continue

    temp = reg_df.dropna(subset=[y_col] + x_cols).copy()

    if len(temp) < 30:
        print(f"{model_name}: sample too small. N={len(temp)}. Skipped.")
        continue

    model, table, fit = run_ols(temp, y_col, x_cols, model_name)

    robust_models[model_name] = model
    robust_tables_list.append(table)
    robust_fits_list.append(fit)

    print(f"\n===== {model_name} =====")
    print(model.summary())

if len(robust_tables_list) > 0:
    robust_ols_tables = pd.concat(robust_tables_list, ignore_index=True)
    robust_ols_fit = pd.DataFrame(robust_fits_list)
    robustness_table_wide = make_regression_wide_table(robust_ols_tables, robust_ols_fit)
else:
    robust_ols_tables = pd.DataFrame()
    robust_ols_fit = pd.DataFrame()
    robustness_table_wide = pd.DataFrame()

print("\nRobustness table, wide format:")
display(robustness_table_wide)

display(robust_ols_fit)


===== R1_raw_return_main_period =====
                                OLS Regression Results                               
Dep. Variable:     raw_cum_ret_202203_202303   R-squared:                       0.178
Model:                                   OLS   Adj. R-squared:                  0.161
Method:                        Least Squares   F-statistic:                     8.342
Date:                       Wed, 27 May 2026   Prob (F-statistic):           2.20e-09
Time:                               17:09:29   Log-Likelihood:                 128.00
No. Observations:                        336   AIC:                            -240.0
Df Residuals:                            328   BIC:                            -209.5
Df Model:                                  7                                         
Covariance Type:                         HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
------------------

model,R1_raw_return_main_period,R2_ew_bank_adjusted_main_period,R3_extended_vw_adjusted_202203_202307,R4_march_2023_vw_adjusted,R5_max_drawdown_main_period
asset_growth,-0.1721**,-0.1861**,-0.1285,-0.1221**,-0.1479**
asset_growth_t,(-2.29),(-2.10),(-1.37),(-2.42),(-2.53)
cash_ratio,0.6426***,0.7573***,0.6522***,0.1651**,0.3039***
cash_ratio_t,(5.03),(5.00),(4.20),(2.21),(3.53)
const,-0.0220,0.1543**,-0.1161,0.0849**,-0.1892***
const_t,(-0.35),(2.09),(-1.44),(2.18),(-4.04)
debt_ratio,-0.1678,-0.2638,-0.2889,0.3470***,-0.0715
debt_ratio_t,(-0.72),(-0.97),(-0.77),(3.32),(-0.30)
equity_ratio,0.3384,0.3586,-0.2401,0.3870**,0.3353
equity_ratio_t,(1.13),(1.00),(-0.68),(2.23),(1.50)


,model,y,N,R2,Adj_R2,F_pvalue
0,R1_raw_return_main_period,raw_cum_ret_202203_202303,336,0.17805,0.16051,0.00000
1,R2_ew_bank_adjusted_main_period,ew_adj_cum_ret_202203_202303,336,0.15652,0.13852,0.00000
2,R3_extended_vw_adjusted_202203_202307,vw_adj_cum_ret_202203_202307,333,0.12158,0.10266,0.00000
3,R4_march_2023_vw_adjusted,vw_adj_cum_ret_202303,333,0.22495,0.20826,0.00000
4,R5_max_drawdown_main_period,max_drawdown_202203_202303,336,0.17213,0.15446,0.00002


In [51]:
# =========================================================
# Cell 22. Commercial bank only robustness: SIC 6020 only
# =========================================================

if "sich_num" in reg_df.columns:
    reg_commercial = reg_df[reg_df["sich_num"].eq(6020)].copy()
else:
    reg_commercial = pd.DataFrame()

reg_commercial_main = reg_commercial.dropna(subset=[main_y] + x_vars_main).copy()

print("Commercial bank only sample:", reg_commercial_main.shape)

commercial_models = {}
commercial_tables = pd.DataFrame()
commercial_fit = pd.DataFrame()
commercial_table_wide = pd.DataFrame()

if len(reg_commercial_main) >= 30:
    c_ols1, c_table1, c_fit1 = run_ols(
        reg_commercial_main,
        main_y,
        x_model1,
        "C1_commercial_basic"
    )

    c_ols2, c_table2, c_fit2 = run_ols(
        reg_commercial_main,
        main_y,
        x_model2,
        "C2_commercial_balance_sheet"
    )

    c_ols3, c_table3, c_fit3 = run_ols(
        reg_commercial_main,
        main_y,
        x_model3,
        "C3_commercial_full"
    )

    commercial_models = {
        "C1_commercial_basic": c_ols1,
        "C2_commercial_balance_sheet": c_ols2,
        "C3_commercial_full": c_ols3
    }

    commercial_tables = pd.concat([c_table1, c_table2, c_table3], ignore_index=True)
    commercial_fit = pd.DataFrame([c_fit1, c_fit2, c_fit3])
    commercial_table_wide = make_regression_wide_table(commercial_tables, commercial_fit)

    print("\nCommercial bank only regression table:")
    display(commercial_table_wide)
    display(commercial_fit)

else:
    print("Commercial bank only regression skipped because sample is too small.")

Commercial bank only sample: (279, 34)

Commercial bank only regression table:


model,C1_commercial_basic,C2_commercial_balance_sheet,C3_commercial_full
asset_growth,NaN,-0.2018*,-0.1202
asset_growth_t,NaN,(-1.78),(-1.20)
cash_ratio,NaN,0.6797***,0.7219***
cash_ratio_t,NaN,(3.68),(4.13)
const,0.0634,0.0453,0.0835
const_t,(0.65),(0.43),(0.79)
debt_ratio,NaN,-0.5207,-0.5446
debt_ratio_t,NaN,(-1.19),(-1.27)
equity_ratio,1.4880**,1.8364***,1.2468*
equity_ratio_t,(2.06),(2.62),(1.77)


,model,y,N,R2,Adj_R2,F_pvalue
0,C1_commercial_basic,vw_adj_cum_ret_202203_202303,279,0.06488,0.05468,0.00827
1,C2_commercial_balance_sheet,vw_adj_cum_ret_202203_202303,279,0.14667,0.12784,0.00000
2,C3_commercial_full,vw_adj_cum_ret_202203_202303,279,0.18390,0.16282,0.00000


In [52]:
# =========================================================
# Cell 23. Summary statistics and correlation table
# =========================================================

summary_vars = [main_y] + x_vars_main

summary_stats = reg_main[summary_vars].describe().T
summary_stats["missing_count"] = reg_main[summary_vars].isna().sum()
summary_stats["missing_ratio"] = reg_main[summary_vars].isna().mean()

corr_matrix = reg_main[summary_vars].corr()

print("Summary Statistics")
display(summary_stats)

print("Correlation Matrix")
display(corr_matrix)

Summary Statistics


,count,mean,std,min,25%,50%,75%,max,missing_count,missing_ratio
vw_adj_cum_ret_202203_202303,336.00000,-0.05043,0.20415,-0.75413,-0.18053,-0.04839,0.07535,0.40138,0,0.00000
size,336.00000,8.91627,1.66781,6.24381,7.69725,8.63320,9.73148,14.37665,0,0.00000
equity_ratio,336.00000,0.10723,0.03941,0.05870,0.08827,0.10105,0.11702,0.42227,0,0.00000
roa,336.00000,0.01264,0.00784,-0.00075,0.00937,0.01159,0.01413,0.06581,0,0.00000
cash_ratio,336.00000,0.10954,0.07982,0.00895,0.04692,0.09014,0.14841,0.35697,0,0.00000
debt_ratio,336.00000,0.05173,0.05347,0.00050,0.02198,0.03596,0.06127,0.34349,0,0.00000
asset_growth,336.00000,0.15026,0.19514,-0.06924,0.04761,0.09871,0.17606,1.18327,0,0.00000
market_to_book,336.00000,1.38901,0.65511,0.68879,1.05071,1.21058,1.45542,4.82887,0,0.00000


Correlation Matrix


,vw_adj_cum_ret_202203_202303,size,equity_ratio,roa,cash_ratio,debt_ratio,asset_growth,market_to_book
vw_adj_cum_ret_202203_202303,1.00000,-0.12309,0.12327,0.10357,0.20304,-0.06176,-0.13692,-0.14559
size,-0.12309,1.00000,-0.01509,0.16604,0.12573,0.28974,-0.05044,0.22622
equity_ratio,0.12327,-0.01509,1.00000,0.70686,0.00683,0.14765,-0.02343,0.18091
roa,0.10357,0.16604,0.70686,1.00000,0.05156,0.22523,-0.16153,0.41178
cash_ratio,0.20304,0.12573,0.00683,0.05156,1.00000,-0.01698,0.23500,0.14670
debt_ratio,-0.06176,0.28974,0.14765,0.22523,-0.01698,1.00000,-0.12227,0.11494
asset_growth,-0.13692,-0.05044,-0.02343,-0.16153,0.23500,-0.12227,1.00000,0.14728
market_to_book,-0.14559,0.22622,0.18091,0.41178,0.14670,0.11494,0.14728,1.00000


In [53]:
# =========================================================
# Cell 24. 주요 결과 저장
# =========================================================

saved_files = []

def save_df(df, filename, index=False):
    if df is not None and isinstance(df, pd.DataFrame):
        path = os.path.join(output_folder, filename)
        df.to_csv(path, index=index, encoding="utf-8-sig")
        saved_files.append(path)
        print(f"Saved: {filename}")

def save_txt(text, filename):
    path = os.path.join(output_folder, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    saved_files.append(path)
    print(f"Saved: {filename}")

# Data and diagnostics
# CRSP-FRB Link 관련 저장
save_df(bank_permco_link_all, "00_crsp_frb_link_all_raw.csv")
save_df(bank_permco_link_filtered, "00_crsp_frb_link_active_2021_2023.csv")
save_df(bank_permco_list_raw, "01_crsp_frb_active_one_row_per_permco.csv")
save_df(bank_permco_list_raw, "01_professor_bank_permco_list_raw.csv")
save_df(sample_diagnostics, "02_sample_diagnostics.csv")
save_df(permno_count_by_permco, "03_permno_count_by_permco.csv")
save_df(bank_bench, "04_monthly_bank_benchmark_returns.csv")
save_df(y_main, "05_y_main_202203_202303.csv")
save_df(y_extended, "06_y_extended_202203_202307.csv")
save_df(y_march, "07_y_march_202303.csv")
save_df(permco_x, "08_permco_compustat_accounting_variables_2021.csv")
save_df(analysis_df, "09_analysis_data_permco_level.csv")
save_df(reg_df, "10_regression_data_all.csv")
save_df(reg_main, "11_final_main_regression_sample.csv")

# Tables
save_df(summary_stats, "12_summary_statistics.csv", index=True)
save_df(corr_matrix, "13_correlation_matrix.csv", index=True)
save_df(ols_tables, "14_main_ols_results_long.csv")
save_df(ols_fit, "15_main_ols_fit.csv")
save_df(main_regression_table_wide, "16_main_regression_table_wide.csv", index=True)
save_df(robust_ols_tables, "17_robustness_ols_results_long.csv")
save_df(robust_ols_fit, "18_robustness_ols_fit.csv")
save_df(robustness_table_wide, "19_robustness_table_wide.csv", index=True)
save_df(commercial_tables, "20_commercial_bank_only_results_long.csv")
save_df(commercial_fit, "21_commercial_bank_only_fit.csv")
save_df(commercial_table_wide, "22_commercial_bank_only_table_wide.csv", index=True)

# Full model summaries
save_txt(ols1.summary().as_text(), "23_ols1_basic_full_summary.txt")
save_txt(ols2.summary().as_text(), "24_ols2_balance_sheet_full_summary.txt")
save_txt(ols3.summary().as_text(), "25_ols3_full_model_summary.txt")

for name, model in robust_models.items():
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", name)
    save_txt(model.summary().as_text(), f"26_{safe_name}_summary.txt")

# ZIP
zip_path = os.path.join(output_folder, "permco_bank_analysis_results.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in saved_files:
        if os.path.exists(file):
            zf.write(file, arcname=os.path.basename(file))

print("\nZIP saved:", zip_path)
print("Number of saved files:", len(saved_files))

Saved: 00_crsp_frb_link_all_raw.csv
Saved: 00_crsp_frb_link_active_2021_2023.csv
Saved: 01_crsp_frb_active_one_row_per_permco.csv
Saved: 01_professor_bank_permco_list_raw.csv
Saved: 02_sample_diagnostics.csv
Saved: 03_permno_count_by_permco.csv
Saved: 04_monthly_bank_benchmark_returns.csv
Saved: 05_y_main_202203_202303.csv
Saved: 06_y_extended_202203_202307.csv
Saved: 07_y_march_202303.csv
Saved: 08_permco_compustat_accounting_variables_2021.csv
Saved: 09_analysis_data_permco_level.csv
Saved: 10_regression_data_all.csv
Saved: 11_final_main_regression_sample.csv
Saved: 12_summary_statistics.csv
Saved: 13_correlation_matrix.csv
Saved: 14_main_ols_results_long.csv
Saved: 15_main_ols_fit.csv
Saved: 16_main_regression_table_wide.csv
Saved: 17_robustness_ols_results_long.csv
Saved: 18_robustness_ols_fit.csv
Saved: 19_robustness_table_wide.csv
Saved: 20_commercial_bank_only_results_long.csv
Saved: 21_commercial_bank_only_fit.csv
Saved: 22_commercial_bank_only_table_wide.csv
Saved: 23_ols1_bas

In [54]:
# =========================================================
# Cell 25. 결과 빠른 해석 자동 생성
# =========================================================

def get_coef_info(table, model_name, variable):
    temp = table[(table["model"] == model_name) & (table["variable"] == variable)].copy()
    if len(temp) == 0:
        return None
    row = temp.iloc[0]
    return {
        "coef": row["coef"],
        "t": row["t_stat"],
        "p": row["p_value"],
        "stars": row["stars"]
    }

key_vars = ["cash_ratio", "asset_growth", "debt_ratio", "market_to_book", "equity_ratio", "roa", "size"]

interpret_rows = []

for var in key_vars:
    info = get_coef_info(ols_tables, "M3_full", var)
    if info is not None:
        interpret_rows.append({
            "variable": var,
            "coef": info["coef"],
            "t_stat": info["t"],
            "p_value": info["p"],
            "stars": info["stars"],
            "direction": "positive" if info["coef"] > 0 else "negative"
        })

key_result_summary = pd.DataFrame(interpret_rows)

print("Key results from M3_full:")
display(key_result_summary)

# 간단한 문장 생성
for _, row in key_result_summary.iterrows():
    var = row["variable"]
    direction = row["direction"]
    stars = row["stars"]
    sig_text = "statistically significant" if stars != "" else "not statistically significant"
    print(f"{var}: {direction}, {sig_text}, coef={row['coef']:.4f}, t={row['t_stat']:.2f}, p={row['p_value']:.4f}")

Key results from M3_full:


,variable,coef,t_stat,p_value,stars,direction
0,cash_ratio,0.71081,4.87448,0.00000,***,positive
1,asset_growth,-0.17312,-2.05163,0.04021,**,negative
2,debt_ratio,-0.21886,-0.85910,0.39029,,negative
3,market_to_book,-0.05961,-2.13771,0.03254,**,negative
4,equity_ratio,0.36066,1.05929,0.28947,,positive
5,roa,3.28135,1.72792,0.08400,*,positive
6,size,-0.01547,-2.29299,0.02185,**,negative


cash_ratio: positive, statistically significant, coef=0.7108, t=4.87, p=0.0000
asset_growth: negative, statistically significant, coef=-0.1731, t=-2.05, p=0.0402
debt_ratio: negative, not statistically significant, coef=-0.2189, t=-0.86, p=0.3903
market_to_book: negative, statistically significant, coef=-0.0596, t=-2.14, p=0.0325
equity_ratio: positive, not statistically significant, coef=0.3607, t=1.06, p=0.2895
roa: positive, statistically significant, coef=3.2814, t=1.73, p=0.0840
size: negative, statistically significant, coef=-0.0155, t=-2.29, p=0.0218


In [55]:
# =========================================================
# Cell 26. 교수님 미팅용 1페이지 요약 자동 생성
# =========================================================

def coef_sentence(var, label):
    info = get_coef_info(ols_tables, "M3_full", var)
    if info is None:
        return f"- {label}: not available in the full model."
    direction = "positive" if info["coef"] > 0 else "negative"
    sig = "statistically significant" if info["stars"] != "" else "not statistically significant"
    return f"- {label}: {direction} coefficient ({info['coef']:.4f}), t = {info['t']:.2f}, p = {info['p']:.4f}; {sig}."

meeting_summary = f"""
PERMCO-Based Bank Analysis Summary
==================================

1. Purpose
----------
Following the professor's instruction, I reconstructed the bank sample using the professor-provided CRSP PERMCO bank list. I then repeated the same analysis using PERMCO-level bank stock returns and 2021 pre-shock Compustat accounting variables.

2. Sample Construction
----------------------
- Professor-provided bank PERMCO list: {len(bank_permcos):,} unique PERMCOs.
- CRSP bank PERMCOs with monthly return data during the shock window: {permco_monthly['PERMCO'].nunique():,}.
- PERMCOs with main-period return outcome from March 2022 to March 2023: {y_main['PERMCO'].nunique():,}.
- PERMCOs matched to Compustat 2021 accounting variables: {permco_x['PERMCO'].nunique():,}.
- Final main regression sample: {reg_main['PERMCO'].nunique():,} PERMCOs.

3. Main Outcome Variable
------------------------
The main dependent variable is the value-weighted bank-benchmark-adjusted cumulative return from March 2022 to March 2023. Returns are adjusted for delisting returns when available. The bank benchmark is constructed using PERMCO-level bank returns.

4. Main Regression
------------------
The full model includes:
Size, EquityRatio, ROA, CashRatio, DebtRatio, AssetGrowth, and MarketToBook.

5. Key Full-Model Results
-------------------------
{coef_sentence("cash_ratio", "CashRatio")}
{coef_sentence("asset_growth", "AssetGrowth")}
{coef_sentence("debt_ratio", "DebtRatio")}
{coef_sentence("market_to_book", "MarketToBook")}
{coef_sentence("equity_ratio", "EquityRatio")}

6. Interpretation
-----------------
The main question is whether pre-shock bank balance sheet characteristics explain cross-sectional differences in bank stock performance during the 2022-2023 interest rate shock. The PERMCO-based analysis directly follows the professor's instruction by identifying banks at the CRSP company level rather than relying only on SIC/CUSIP-based sample construction.

7. Tables Prepared
------------------
- Sample diagnostics table
- Summary statistics
- Correlation matrix
- Main regression table
- Robustness regression table
- Commercial-bank-only robustness table, if sample size permits

8. Files Saved
--------------
All tables and regression outputs are saved in:
{output_folder}
"""

print(meeting_summary)

save_txt(meeting_summary, "27_meeting_summary_permco_analysis.txt")


PERMCO-Based Bank Analysis Summary

1. Purpose
----------
Following the professor's instruction, I reconstructed the bank sample using the professor-provided CRSP PERMCO bank list. I then repeated the same analysis using PERMCO-level bank stock returns and 2021 pre-shock Compustat accounting variables.

2. Sample Construction
----------------------
- Professor-provided bank PERMCO list: 394 unique PERMCOs.
- CRSP bank PERMCOs with monthly return data during the shock window: 362.
- PERMCOs with main-period return outcome from March 2022 to March 2023: 345.
- PERMCOs matched to Compustat 2021 accounting variables: 354.
- Final main regression sample: 336 PERMCOs.

3. Main Outcome Variable
------------------------
The main dependent variable is the value-weighted bank-benchmark-adjusted cumulative return from March 2022 to March 2023. Returns are adjusted for delisting returns when available. The bank benchmark is constructed using PERMCO-level bank returns.

4. Main Regression
--------

In [56]:
# =========================================================
# Cell 27. ChatGPT에게 결과 보여줄 때 복붙용
# =========================================================

def df_to_markdown_safe(df, max_rows=30):
    if df is None or len(df) == 0:
        return "EMPTY"
    try:
        return df.head(max_rows).to_markdown()
    except:
        return df.head(max_rows).to_string()

gpt_report = f"""
I reran the bank fragility analysis using the professor-provided PERMCO bank list.

========================
1. SAMPLE DIAGNOSTICS
========================

{df_to_markdown_safe(sample_diagnostics, 20)}

========================
2. FINAL REGRESSION SAMPLE
========================

reg_df shape: {reg_df.shape}
reg_main shape: {reg_main.shape}
Unique PERMCO in reg_main: {reg_main['PERMCO'].nunique()}
Unique gvkey in reg_main: {reg_main['gvkey'].nunique()}

========================
3. SIC DISTRIBUTION
========================

{df_to_markdown_safe(reg_main['sich_num'].value_counts(dropna=False).sort_index().reset_index().rename(columns={'index':'sich_num', 'sich_num':'N'}), 50)}

========================
4. SUMMARY STATISTICS
========================

{df_to_markdown_safe(summary_stats, 50)}

========================
5. CORRELATION MATRIX
========================

{df_to_markdown_safe(corr_matrix, 50)}

========================
6. MAIN REGRESSION TABLE
========================

{df_to_markdown_safe(main_regression_table_wide, 80)}

========================
7. MAIN MODEL FIT
========================

{df_to_markdown_safe(ols_fit, 20)}

========================
8. ROBUSTNESS TABLE
========================

{df_to_markdown_safe(robustness_table_wide, 100)}

========================
9. ROBUSTNESS MODEL FIT
========================

{df_to_markdown_safe(robust_ols_fit, 50)}

========================
10. COMMERCIAL BANK ONLY TABLE
========================

{df_to_markdown_safe(commercial_table_wide, 80)}

========================
11. COMMERCIAL BANK ONLY MODEL FIT
========================

{df_to_markdown_safe(commercial_fit, 20)}

========================
12. KEY FULL MODEL COEFFICIENTS
========================

{df_to_markdown_safe(key_result_summary, 30)}

========================
13. TASK FOR CHATGPT
========================

Please interpret these PERMCO-based results for my meeting with the professor.
Focus on:
1. whether the PERMCO-based sample construction worked,
2. whether the main liquidity / asset growth / market-to-book results are consistent with the earlier analysis,
3. whether the results are strong enough to discuss with the professor,
4. what tables I should bring to the meeting,
5. what weaknesses or caveats I should mention,
6. what next empirical step I should take.
"""

print(gpt_report)

# 저장도 해둠
gpt_report_path = os.path.join(output_folder, "28_COPY_THIS_TO_CHATGPT.txt")
with open(gpt_report_path, "w", encoding="utf-8") as f:
    f.write(gpt_report)

print("\nSaved copy-paste report to:")
print(gpt_report_path)


I reran the bank fragility analysis using the professor-provided PERMCO bank list.

1. SAMPLE DIAGNOSTICS

|    | step                                              |   N_PERMCO |   N_rows |
|---:|:--------------------------------------------------|-----------:|---------:|
|  0 | Raw CRSP-FRB Link file                            |       1450 |     1495 |
|  1 | CRSP-FRB links active during 2021-2023            |        394 |      395 |
|  2 | One row per active PERMCO bank list               |        394 |      394 |
|  3 | CRSP bank PERMCOs with monthly return data        |        362 |     5898 |
|  4 | PERMCOs with main-period Y variable               |        345 |      345 |
|  5 | PERMCOs matched to Compustat 2021 accounting data |        354 |      354 |
|  6 | Final main regression sample                      |        336 |      336 |

2. FINAL REGRESSION SAMPLE

reg_df shape: (345, 34)
reg_main shape: (336, 34)
Unique PERMCO in reg_main: 336
Unique gvkey in reg_main: 336

3. S